# CS781 Legal NER — Clean Experiment Notebook

**Purpose:** one clean, reproducible notebook for checkpoint/data inventory, a locked exact-span evaluation harness, RESP/WIT diagnostics, decoding-bias diagnostics, and safe candidate comparisons.

## Safety and experiment policy
- Uses the best established NER checkpoint (`champion_recovery.pt`) as the champion and the Phase-1 checkpoint as its encoder source.
- Searches all attached Kaggle datasets for backup datasets/checkpoints and prints what is present.
- Other checkpoints are inventoried and compatibility-checked; a filename containing `latest` does **not** automatically make it the best NER checkpoint.
- No training occurs in this notebook. Earlier short fine-tuning and frozen-probe experiments regressed, so training is deferred until the diagnostics identify a specific hypothesis.
- Never overwrites `prediction_champion_plus_rules.json`, any checkpoint, or any existing prediction file.
- Fold 1 is the training/development fold; fold 2 is a repeatedly used validation/development fold; fold 3 is test and must not be used for tuning.
- Exact-span metrics are reported for the prediction pipeline actually run. The threshold-only baseline is not falsely presented as the champion-plus-rules score of 0.299526.

In [ ]:
# 1. Necessary imports only + runtime
import os, json, re, gc, math, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2Model
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Inventory all available datasets/checkpoints before choosing anything
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
CHECKPOINT_NAMES = {
    "champion_recovery.pt", "best_ner_model.pt", "checkpoint_latest.pt",
    "targeted_pilot_8_steps.pt", "best_model.pt", "model.pt"
}
DATASET_NAMES = {
    "legal_bpe_32k.json", "fold1-text.json", "fold1-gold.json",
    "fold2-text.json", "fold2-gold.json", "fold3-text.json",
    "train.json", "test.json"
}
all_files = []
for root in ROOTS:
    if root.exists():
        all_files.extend(p for p in root.rglob("*") if p.is_file())

all_files = sorted(set(all_files))
checkpoint_files = [p for p in all_files if p.name in CHECKPOINT_NAMES or p.suffix.lower() in {".pt", ".pth"}]
dataset_files = [p for p in all_files if p.name in DATASET_NAMES or ("fold" in p.name.lower() and p.suffix.lower() == ".json")]

print("=== CHECKPOINT INVENTORY ===")
if checkpoint_files:
    for p in checkpoint_files:
        print(f"{p}  [{p.stat().st_size / 1024**2:.1f} MiB]")
else:
    print("No checkpoint files found.")

print("\n=== DATASET/TOKENIZER INVENTORY ===")
for p in dataset_files:
    print(p)

print("\n=== BACKUP DATASET DIRECTORIES ===")
for p in sorted({str(x.parent) for x in dataset_files}):
    print(p)

# Pick the recovery champion explicitly; don't accidentally choose an arbitrary checkpoint_latest.pt.
champion_candidates = [p for p in checkpoint_files if p.name == "champion_recovery.pt"]
if not champion_candidates:
    champion_candidates = [p for p in checkpoint_files if p.name == "best_ner_model.pt"]
if not champion_candidates:
    raise FileNotFoundError(
        "No champion_recovery.pt or best_ner_model.pt found under /kaggle/input or /kaggle/working. "
        "Attach the CS781 recovery/checkpoint dataset in Kaggle."
    )

def rank_champion_path(p):
    s = str(p).lower()
    return (
        0 if "cs781_recovery" in s else 1,
        0 if "working-backup" in s else 1,
        0 if "/kaggle/input/" in s else 1,
        s
    )

CHAMPION_PATH = sorted(champion_candidates, key=rank_champion_path)[0]

phase1_candidates = [
    p for p in checkpoint_files
    if p.name == "checkpoint_latest.pt" and "cs781-phase-1-checkpoints" in str(p)
]
if not phase1_candidates:
    phase1_candidates = [
        p for p in checkpoint_files
        if p.name == "checkpoint_latest.pt" and "phase-1" in str(p).lower()
    ]
if not phase1_candidates:
    phase1_candidates = [p for p in checkpoint_files if p.name == "checkpoint_latest.pt"]

if not phase1_candidates:
    raise FileNotFoundError("No Phase-1 checkpoint_latest.pt found.")

PHASE1_PATH = sorted(phase1_candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

print("\nSELECTED NER CHAMPION:", CHAMPION_PATH)
print("SELECTED PHASE-1 ENCODER:", PHASE1_PATH)
print("Selection policy: recovery champion for NER; Phase-1 checkpoint for encoder.")

In [ ]:
# 3. Inspect checkpoint metadata; only the known champion is selected for NER inference
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

recovery = safe_torch_load(CHAMPION_PATH)
phase1_ckpt = safe_torch_load(PHASE1_PATH)

print("Champion checkpoint keys:", list(recovery.keys()) if isinstance(recovery, dict) else type(recovery))
print("Phase-1 checkpoint keys:", list(phase1_ckpt.keys()) if isinstance(phase1_ckpt, dict) else type(phase1_ckpt))
if isinstance(phase1_ckpt, dict):
    print("Phase-1 step/epoch metadata:", {
        k: phase1_ckpt.get(k) for k in ("step", "global_step", "epoch", "best_step")
        if k in phase1_ckpt
    })

# Report other checkpoints' metadata without silently replacing the champion.
print("\n=== OTHER CHECKPOINT METADATA ===")
for p in checkpoint_files:
    if p in (CHAMPION_PATH, PHASE1_PATH):
        continue
    try:
        obj = safe_torch_load(p)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            meta = {k: obj.get(k) for k in ("step", "global_step", "epoch", "optimizer_steps")
                    if k in obj and isinstance(obj.get(k), (int, float, str))}
            state = obj.get("model_state_dict")
            shape_info = None
            if isinstance(state, dict):
                cls = state.get("classifier.weight")
                if cls is not None and hasattr(cls, "shape"):
                    shape_info = tuple(cls.shape)
            print(f"{p}: keys={keys[:12]}, metadata={meta}, classifier.weight={shape_info}")
        else:
            print(f"{p}: object={type(obj).__name__}")
        del obj
    except Exception as e:
        print(f"{p}: could not inspect ({type(e).__name__}: {e})")

In [ ]:
# 4. Reconstruct model and load tokenizer/data files
BIO_LABELS = list(recovery["bio_labels"])
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
assert len(BIO_LABELS) == 25, f"Expected 25 BIO labels, got {len(BIO_LABELS)}"

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase-1 checkpoint has no model_config; cannot reconstruct encoder safely.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(pretrained_lm.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(outputs.last_hidden_state))

phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)
lm_wrapper = type("LMWrapper", (), {"transformer": phase1_lm, "config": gpt2_config})()
champion_model = LegalNERModel(lm_wrapper, len(BIO_LABELS)).to(device)
champion_model.load_state_dict(recovery["model_state_dict"], strict=True)
champion_model.eval()

def find_named_file(filename, preferred_tokens=()):
    matches = [p for p in all_files if p.name == filename]
    if not matches:
        raise FileNotFoundError(f"Required file {filename} not found in Kaggle input/working.")
    preferred = [p for p in matches if all(t.lower() in str(p).lower() for t in preferred_tokens)]
    candidates = preferred or matches
    return sorted(candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

tokenizer_path = find_named_file("legal_bpe_32k.json")
fold1_text_path = find_named_file("fold1-text.json")
fold1_gold_path = find_named_file("fold1-gold.json")
fold2_text_path = find_named_file("fold2-text.json")
fold2_gold_path = find_named_file("fold2-gold.json")
fold3_text_path = find_named_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

# Normalize document IDs to strings consistently.
train_texts = {str(k): v for k, v in train_texts.items()}
train_gold = {str(k): v for k, v in train_gold.items()}
val_texts = {str(k): v for k, v in val_texts.items()}
val_gold = {str(k): v for k, v in val_gold.items()}
test_texts = {str(k): v for k, v in test_texts.items()}

print("Champion loaded:", CHAMPION_PATH)
print("Tokenizer:", tokenizer_path, "| vocab size:", tokenizer.get_vocab_size())
print("Documents: train", len(train_texts), "| validation", len(val_texts), "| test", len(test_texts))
print("Train/val IDs disjoint:", set(train_texts).isdisjoint(val_texts))
print("Validation text/gold IDs match:", set(val_texts) == set(val_gold))
print("Train text/gold IDs match:", set(train_texts) == set(train_gold))
assert set(train_texts) == set(train_gold)
assert set(val_texts) == set(val_gold)

In [ ]:
# 5. Find and parse champion thresholds
threshold_candidates = [
    p for p in all_files
    if p.suffix.lower() == ".json" and "threshold" in p.name.lower()
]
print("Threshold files found:")
for p in threshold_candidates:
    print(" ", p)

def extract_thresholds(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("champ_thr", "thresholds", "label_thresholds", "threshold_by_label"):
        if isinstance(obj.get(key), dict) and obj[key] and all(isinstance(v, (int, float)) for v in obj[key].values()):
            return {str(k): float(v) for k, v in obj[key].items()}
    if obj and all(isinstance(v, (int, float)) for v in obj.values()):
        return {str(k): float(v) for k, v in obj.items()}
    return None

champ_thr = None
threshold_path_used = None
for p in sorted(threshold_candidates, key=lambda p: (
    0 if "champion_thresholds_recovery" in p.name.lower() else 1,
    0 if "cs781_recovery" in str(p).lower() else 1,
    str(p)
)):
    try:
        obj = load_json(p)
        found = extract_thresholds(obj)
        if found and all(lab in found for lab in ENTITY_LABELS):
            champ_thr, threshold_path_used = found, p
            break
    except Exception:
        pass

if champ_thr is None:
    # These are the documented recovery thresholds from the existing notebook.
    champ_thr = {
        "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
        "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
        "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35
    }
    print("WARNING: no usable threshold JSON found; using documented recovery thresholds.")
else:
    print("Loaded thresholds from:", threshold_path_used)
print("Thresholds:", champ_thr)

In [ ]:
# 6. Core inference, BIO decoding, and exact-span evaluation helpers
from bisect import bisect_left, bisect_right

NUM_LABELS = len(BIO_LABELS)
legal_start = np.array([tag == "O" or tag.startswith("B-") for tag in BIO_LABELS], dtype=bool)
legal_transition = np.ones((NUM_LABELS, NUM_LABELS), dtype=bool)
for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            legal_transition[prev_id, curr_id] = prev_tag in (f"B-{curr_tag[2:]}", f"I-{curr_tag[2:]}")

def bio_viterbi_decode(logits):
    emissions = logits.detach().float().cpu().numpy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float64)
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] emissions, got {emissions.shape}")
    if len(emissions) == 0:
        return []
    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12
    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    back = np.zeros(emissions.shape, dtype=np.int32)
    scores[0, legal_start] = emissions[0, legal_start]
    for t in range(1, len(emissions)):
        cand = np.where(legal_transition, scores[t-1][:, None], neg_inf) + emissions[t][None, :]
        back[t] = np.argmax(cand, axis=0)
        scores[t] = cand[back[t], np.arange(NUM_LABELS)]
    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return list(reversed(path))

def decode_spans(text, offsets, pred_ids):
    spans, current_label, current_start, current_end = [], None, None, None
    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            s, e = current_start, current_end
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e-1].isspace(): e -= 1
            if s < e: spans.append((s, e, current_label))
        current_label = current_start = current_end = None
    for (s, e), pred_id in zip(offsets, pred_ids):
        if e <= s:
            continue
        tag = BIO_LABELS[int(pred_id)]
        if tag == "O":
            close(); continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or current_label != lab:
            close(); current_label, current_start, current_end = lab, s, e
        else:
            current_end = e
    close()
    return sorted(set(spans))

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448
@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)
    if not n:
        return offsets, torch.zeros((0, len(BIO_LABELS)))
    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    counts = torch.zeros(n, dtype=torch.float32)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        logits = model(x, torch.ones_like(x))[0].float().cpu()
        logit_sum[start:end] += logits
        counts[start:end] += 1
        if end == n: break
    avg_logits = logit_sum / counts.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg_logits, dim=-1)

def predict_texts(model, texts):
    result = {}
    model.eval()
    for doc_id, text in tqdm(texts.items(), desc="Predicting", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets):
            result[doc_id] = []
            continue
        pred_ids = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))), dtype=torch.long)
        starts, ends = [a for a, b in offsets], [b for a, b in offsets]
        spans = []
        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ix = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            conf = float(probs[ix, pred_ids[ix]].mean()) if ix else 0.0
            spans.append((int(s), int(e), lab, conf))
        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {d: [(s, e, lab) for s, e, lab, conf in spans if conf >= thresholds.get(lab, 0.0)] for d, spans in raw.items()}

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels=ENTITY_LABELS):
    metrics = {}
    for lab in labels:
        gold_set, pred_set = set(), set()
        for d, ents in gold_by_doc.items():
            for ent in ents:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    gold_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        for d, spans in pred_by_doc.items():
            for ent in spans:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    pred_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        tp = len(gold_set & pred_set); fp = len(pred_set - gold_set); fn = len(gold_set - pred_set)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        metrics[lab] = {"gold":len(gold_set), "pred":len(pred_set), "TP":tp, "FP":fp, "FN":fn,
                        "precision":p, "recall":r, "f1":f}
    metrics["MACRO_F1"] = sum(metrics[x]["f1"] for x in labels) / len(labels)
    return metrics

# Verify exact recovery weights and a forward pass.
state = recovery["model_state_dict"]
loaded = champion_model.state_dict()
assert set(state) == set(loaded)
max_diff = max((loaded[k].detach().cpu() - state[k]).abs().max().item() for k in state)
assert max_diff == 0.0
with torch.inference_mode():
    z = champion_model(torch.zeros((1, 8), dtype=torch.long, device=device), torch.ones((1, 8), dtype=torch.long, device=device))
assert tuple(z.shape) == (1, 8, len(BIO_LABELS))
print("Champion weights verified exactly; output shape:", tuple(z.shape))

In [ ]:
# 7. Baseline: raw champion and thresholded champion, using the SAME evaluator
raw_val = predict_texts(champion_model, val_texts)
threshold_val = apply_thr(raw_val, champ_thr)
raw_metrics = calculate_exact_span_metrics(val_gold, {
    d: [(s,e,l) for s,e,l,c in spans] for d, spans in raw_val.items()
})
threshold_metrics = calculate_exact_span_metrics(val_gold, threshold_val)

def print_metrics(title, m):
    print("\n" + title, "| macro-F1 =", round(m["MACRO_F1"], 6))
    print(f"{'LABEL':12} {'P':>7} {'R':>7} {'F1':>7} {'TP':>5} {'FP':>5} {'FN':>5} {'PRED':>6}")
    for lab in ENTITY_LABELS:
        x = m[lab]
        print(f"{lab:12} {x['precision']:7.3f} {x['recall']:7.3f} {x['f1']:7.3f} {x['TP']:5d} {x['FP']:5d} {x['FN']:5d} {x['pred']:6d}")

print_metrics("RAW CHAMPION", raw_metrics)
print_metrics("THRESHOLD-ONLY CHAMPION", threshold_metrics)
print("\nReference from prior established champion+rules pipeline: 0.299526")
print("Important: threshold-only score is NOT expected to equal the champion+rules reference.")

In [ ]:
# 8. Annotation/data audit: counts, document coverage, concentration, role-context examples
def label_audit(texts, gold, labels=("RESP", "WIT")):
    for lab in labels:
        doc_counts, surface_counts = Counter(), Counter()
        examples = []
        for doc_id, ents in gold.items():
            for ent in ents:
                if len(ent) < 3 or str(ent[2]) != lab: continue
                s, e = int(ent[0]), int(ent[1])
                doc_counts[str(doc_id)] += 1
                surface_counts[texts[str(doc_id)][s:e].strip()] += 1
                if len(examples) < 8:
                    text = texts[str(doc_id)]
                    examples.append((str(doc_id), s, e, text[max(0,s-100):s], text[s:e], text[e:min(len(text),e+120)]))
        total = sum(doc_counts.values())
        print(f"\n=== {lab}: {total} spans in {len(doc_counts)} documents ===")
        print("Top documents:", doc_counts.most_common(8))
        print("Largest document share:", round(max(doc_counts.values(), default=0)/max(total,1), 3))
        print("Top annotated surfaces:", surface_counts.most_common(12))
        print("Examples (context-before | annotated-span | context-after):")
        for ex in examples:
            print(f"doc={ex[0]} [{ex[1]}:{ex[2]}] ...{ex[3]!r} [[{ex[4]!r}]] {ex[5]!r}")

print("TRAIN LABEL COUNTS")
print(Counter(str(e[2]) for ents in train_gold.values() for e in ents if len(e) >= 3))
label_audit(train_texts, train_gold)
label_audit(val_texts, val_gold)

In [ ]:
# 9. Token-level diagnostic: what labels compete at gold RESP/WIT positions?
from collections import defaultdict

def gold_token_probability_audit(model, texts, gold, labels=("RESP", "WIT")):
    summaries = defaultdict(lambda: {"probs": [], "top": Counter(), "B_top": 0, "I_top": 0, "B_n": 0, "I_n": 0})
    for doc_id, text in tqdm(texts.items(), desc="Gold-token audit", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets): continue
        top_ids = probs.argmax(-1).tolist()
        for ann in gold[str(doc_id)]:
            if len(ann) < 3: continue
            s, e, lab = int(ann[0]), int(ann[1]), str(ann[2])
            if lab not in labels: continue
            ix = [i for i,(a,b) in enumerate(offsets) if b > a and a < e and b > s]
            for j, i in enumerate(ix):
                target_tag = ("B-" if j == 0 else "I-") + lab
                tid = label2id[target_tag]
                summaries[lab]["probs"].append(float(probs[i, tid]))
                summaries[lab]["top"][BIO_LABELS[top_ids[i]]] += 1
                key = "B" if j == 0 else "I"
                summaries[lab][key+"_n"] += 1
                summaries[lab][key+"_top"] += int(top_ids[i] == tid)
    for lab in labels:
        d = summaries[lab]
        print(f"\n{lab}: mean gold target-tag probability={np.mean(d['probs']) if d['probs'] else float('nan'):.4f}")
        print(f"B top1={d['B_top']}/{d['B_n']} | I top1={d['I_top']}/{d['I_n']}")
        print("Competing top labels:", d["top"].most_common(10))

gold_token_probability_audit(champion_model, train_texts, train_gold)
gold_token_probability_audit(champion_model, val_texts, val_gold)

In [ ]:
# 10. RESP/WIT logit-bias diagnostic on validation only
# Does not train, save, or alter model weights.
bias_values = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
bias_results = {}
for bias in bias_values:
    candidate = {}
    for doc_id, text in tqdm(val_texts.items(), desc=f"Bias {bias}", leave=False):
        offsets, probs = diag_doc_probs(champion_model, text)
        if not len(offsets):
            candidate[doc_id] = []; continue
        logits = torch.log(probs.clamp_min(1e-12))
        logits = logits.clone()
        for lab in ("RESP", "WIT"):
            logits[:, label2id["B-"+lab]] += bias
            logits[:, label2id["I-"+lab]] += bias
        adjusted_probs = torch.softmax(logits, dim=-1)
        pred_ids = bio_viterbi_decode(torch.log(adjusted_probs.clamp_min(1e-12)))
        spans = decode_spans(text, offsets, pred_ids)
        candidate[doc_id] = spans
    metrics = calculate_exact_span_metrics(val_gold, candidate)
    bias_results[bias] = (candidate, metrics)
    print({
        "bias": bias, "macro_f1": round(metrics["MACRO_F1"], 6),
        "RESP_f1": round(metrics["RESP"]["f1"], 4),
        "WIT_f1": round(metrics["WIT"]["f1"], 4),
        "RESP_pred": metrics["RESP"]["pred"], "WIT_pred": metrics["WIT"]["pred"],
        "total_pred": sum(len(v) for v in candidate.values())
    })
print("\nThis is a diagnostic; bias candidates must not replace the champion automatically.")

In [ ]:
# 11. Confidence filtering diagnostic for bias=2 (validation only)
# This isolates whether high-confidence additions can improve the threshold-only baseline.
bias2_candidate, bias2_metrics = bias_results[2.0]
# Recreate adjusted probabilities and span confidence from predicted BIO tags.
bias2_scored = {}
for doc_id, text in tqdm(val_texts.items(), desc="Bias-2 span confidence", leave=False):
    offsets, probs = diag_doc_probs(champion_model, text)
    if not len(offsets):
        bias2_scored[doc_id] = []; continue
    logits = torch.log(probs.clamp_min(1e-12)).clone()
    for lab in ("RESP", "WIT"):
        logits[:, label2id["B-"+lab]] += 2.0
        logits[:, label2id["I-"+lab]] += 2.0
    adj = torch.softmax(logits, dim=-1)
    ids = bio_viterbi_decode(torch.log(adj.clamp_min(1e-12)))
    tag_conf = adj[torch.arange(len(ids)), torch.tensor(ids, dtype=torch.long)].tolist()
    spans = decode_spans(text, offsets, ids)
    scored = []
    for s,e,lab in spans:
        if lab not in ("RESP", "WIT"): continue
        ix = [i for i,(a,b) in enumerate(offsets) if b > a and a < e and b > s]
        conf = float(np.mean([tag_conf[i] for i in ix])) if ix else 0.0
        scored.append((s,e,lab,conf))
    bias2_scored[doc_id] = scored

for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    cand = {d: list(threshold_val.get(d, [])) for d in val_texts}
    for d, spans in bias2_scored.items():
        for s,e,lab,conf in spans:
            if conf < thr: continue
            if any(s < pe and e > ps for ps,pe,_ in cand[d]):
                continue
            cand[d].append((s,e,lab))
    m = calculate_exact_span_metrics(val_gold, cand)
    print({"threshold":thr, "macro_f1":round(m["MACRO_F1"],6),
           "RESP_f1":round(m["RESP"]["f1"],4), "WIT_f1":round(m["WIT"]["f1"],4),
           "RESP_pred":m["RESP"]["pred"], "WIT_pred":m["WIT"]["pred"],
           "total_pred":sum(len(v) for v in cand.values())})
print("\nNote: this hybrid is compared against threshold-only baseline, not the full rule-based champion.")

In [ ]:
# 12. Role-cue audit (not a prediction rule)
# Counts whether legal role language appears near gold RESP/WIT mentions.
ROLE_CUES = {
    "RESP": re.compile(r"\b(respondent|defendant|accused|opposite party|non-applicant)\b", re.I),
    "WIT": re.compile(r"\b(witness|deposed|testified|statement of|examined as a witness)\b", re.I),
    "PETITIONER": re.compile(r"\b(petitioner|applicant|appellant|complainant|plaintiff)\b", re.I),
}
def cue_context_audit(texts, gold, window=300):
    for lab in ("RESP", "WIT"):
        total = near_any = near_same = 0
        examples = []
        for d, ents in gold.items():
            text = texts[d]
            for ent in ents:
                if len(ent) < 3 or str(ent[2]) != lab: continue
                s,e = int(ent[0]),int(ent[1]); total += 1
                ctx = text[max(0,s-window):min(len(text),e+window)]
                any_cue = any(rx.search(ctx) for rx in ROLE_CUES.values())
                same_cue = bool(ROLE_CUES[lab].search(ctx))
                near_any += int(any_cue); near_same += int(same_cue)
                if len(examples) < 5:
                    examples.append((d, text[max(0,s-100):s], text[s:e], text[e:min(len(text),e+100)], any_cue, same_cue))
        print(f"\n{lab}: gold={total}, any role cue within ±{window} chars={near_any} ({near_any/max(total,1):.1%}), same-label cue={near_same} ({near_same/max(total,1):.1%})")
        for x in examples:
            print(f"doc={x[0]} ...{x[1]!r} [[{x[2]!r}]] {x[3]!r} | anycue={x[4]} samecue={x[5]}")
cue_context_audit(train_texts, train_gold)
cue_context_audit(val_texts, val_gold)

In [ ]:
# 13. Checkpoint decision report
# We use the champion recovery checkpoint for inference. Other checkpoints are candidates only
# if their architecture, label order, and validation score can be established explicitly.
print("Current NER checkpoint:", CHAMPION_PATH)
print("Phase-1 encoder checkpoint:", PHASE1_PATH)
print("Champion validation threshold-only macro-F1:", round(threshold_metrics["MACRO_F1"], 6))
print("Best established full champion+rules reference from prior notebook: 0.299526")
print("No checkpoint was modified or saved.")
print("\nTraining is intentionally not run automatically: prior targeted fine-tuning and frozen-head experiments regressed.")
print("Next decision should be based on the annotation examples, role-cue coverage, and validation error analysis above.")

In [ ]:

# CS781 — NER checkpoint comparison: metadata + weight identity
# Read-only: does not change the champion model or write files.

import hashlib

NER_CANDIDATE_PATHS = {
    "champion_recovery": CHAMPION_PATH,
    "persistent_champion": Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-persistent-ner-checkpoints/champion_best_ner_model.pt"
    ),
    "phase2_best": Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-working-backup-20261010-215243/"
        "phase2_ner_corrected/best_ner_model.pt"
    ),
}

def tensor_state_fingerprint(state):
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key]
        if torch.is_tensor(value):
            h.update(key.encode())
            h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()[:16]

candidate_metadata = {}
for name, path in NER_CANDIDATE_PATHS.items():
    if not path.exists():
        print(f"\n{name}: NOT FOUND — {path}")
        continue

    obj = safe_torch_load(path)
    state = obj.get("model_state_dict", {})
    labels = obj.get("bio_labels")
    compatible = (
        isinstance(state, dict)
        and set(state.keys()) == set(champion_model.state_dict().keys())
        and labels == BIO_LABELS
    )

    candidate_metadata[name] = {
        "path": path,
        "checkpoint": obj,
        "compatible": compatible,
        "fingerprint": tensor_state_fingerprint(state) if state else None,
    }

    print(f"\n{name}")
    print("  Path:", path)
    print("  Compatible state + label order:", compatible)
    print("  Recorded epoch:", obj.get("epoch"))
    print("  Recorded validation macro-F1:", obj.get("val_exact_span_macro_f1"))
    print("  Training fold:", obj.get("train_fold"))
    print("  Validation fold:", obj.get("validation_fold"))
    print("  Phase-1 source:", obj.get("phase1_checkpoint"))
    print("  Weight fingerprint:", candidate_metadata[name]["fingerprint"])

print("\n=== WEIGHT IDENTITY ===")
for name, info in candidate_metadata.items():
    print(name, "->", info["fingerprint"])

print("\nMatching fingerprints mean identical tensor weights.")
print("Recorded scores may use different evaluation pipelines; verify them below.")


In [ ]:

# CS781 — Fair validation comparison of NER checkpoints
# Same fold, thresholds, decoder and exact-span evaluator for every candidate.
# No checkpoint is overwritten; champion weights are restored/untouched.

import copy

champion_fingerprint_before = tensor_state_fingerprint(
    champion_model.state_dict()
)

checkpoint_comparison = {}

for name, info in candidate_metadata.items():
    if not info["compatible"]:
        print(f"Skipping {name}: incompatible architecture or BIO label order.")
        continue

    candidate_model = copy.deepcopy(champion_model).to(device)
    candidate_model.load_state_dict(
        info["checkpoint"]["model_state_dict"], strict=True
    )
    candidate_model.eval()

    raw_candidate = predict_texts(candidate_model, val_texts)
    filtered_candidate = apply_thr(raw_candidate, champ_thr)

    candidate_metrics = calculate_exact_span_metrics(
        val_gold, filtered_candidate, ENTITY_LABELS
    )

    checkpoint_comparison[name] = {
        "macro_f1": candidate_metrics["MACRO_F1"],
        "metrics": candidate_metrics,
        "fingerprint": info["fingerprint"],
        "path": str(info["path"]),
    }

    print(
        f"\n{name}: threshold-only macro-F1 = "
        f"{candidate_metrics['MACRO_F1']:.6f}"
    )
    for lab in ENTITY_LABELS:
        m = candidate_metrics[lab]
        print(
            f"  {lab:12s} F1={m['f1']:.4f} "
            f"P={m['precision']:.4f} R={m['recall']:.4f} "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']}"
        )

    del candidate_model, raw_candidate, filtered_candidate
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n=== CHECKPOINT RANKING: SAME THRESHOLD-ONLY PIPELINE ===")
for name, result in sorted(
    checkpoint_comparison.items(),
    key=lambda item: item[1]["macro_f1"],
    reverse=True
):
    print(
        f"{name:24s} "
        f"macro-F1={result['macro_f1']:.6f} "
        f"fingerprint={result['fingerprint']}"
    )

champion_fingerprint_after = tensor_state_fingerprint(
    champion_model.state_dict()
)
assert champion_fingerprint_before == champion_fingerprint_after
print("\nChampion weights unchanged:", True)
print("No files were written.")
print(
    "\nReminder: these are threshold-only scores. "
    "Do not compare them directly with the full champion-plus-rules "
    "reference of 0.299526 until the rules are restored."
)


In [ ]:

# CS781 — inspect existing rule pipeline in memory
# Read-only: does not modify model weights or saved predictions.

import inspect

_names = [
    "counsel_spans",
    "merge_all",
    "keep",
    "chosen",
    "champ_thr",
    "as_scored",
    "calculate_exact_span_metrics",
]

for _name in _names:
    print(f"\n--- {_name} ---")
    if _name not in globals():
        print("NOT FOUND in current notebook globals")
        continue

    _obj = globals()[_name]
    print("Type:", type(_obj).__name__)

    if callable(_obj):
        try:
            print(inspect.getsource(_obj))
        except (OSError, TypeError):
            print("Source unavailable; callable is:", _obj)
    else:
        print(repr(_obj))


In [ ]:

# CS781 — Standalone counsel + judge candidate rules
# No training, checkpoint modifications, or file writes.

import re
from collections import Counter

assert all(x in globals() for x in [
    "train_texts", "train_gold", "val_texts", "val_gold",
    "champion_model", "predict_texts", "apply_thr",
    "calculate_exact_span_metrics", "ENTITY_LABELS"
])

# ---- Counsel name candidates ----
_CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
_INIT = r"(?:[A-Z]\.?\s*){1,4}"
_NAME_WORD = rf"(?:{_INIT})?{_CAP}"
_COUNSEL_NAME = rf"{_NAME_WORD}(?:[ \t]+{_NAME_WORD}){{0,3}}"
_HONORIFIC = r"(?:Mr|Mrs|Ms|Miss|Dr|Shri|Sri|Smt)\.?"

_HON_NAME_RE = re.compile(
    rf"(?<!\w){_HONORIFIC}[ \t]+({_COUNSEL_NAME})",
    re.IGNORECASE
)

def counsel_candidates(text):
    """Return candidate name spans; labels are assigned separately."""
    return sorted({
        (int(m.start(1)), int(m.end(1)))
        for m in _HON_NAME_RE.finditer(text)
    })

# ---- Contextual party-side assignment ----
_PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
_PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
_FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
_MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

_CUE_A = re.compile(rf"\b{_FOR}\s+(?:the\s+)?{_MODS}{_PARTY_A}", re.I)
_CUE_R = re.compile(rf"\b{_FOR}\s+(?:the\s+)?{_MODS}{_PARTY_R}", re.I)
_HDR_A = re.compile(rf"{_PARTY_A}[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
_HDR_R = re.compile(rf"{_PARTY_R}[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
_HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$", re.I)

def assign_counsel_side(text, s, e, previous=None):
    # Follow the V5 cue order: header, right context, left context,
    # then nearby previous assignment.
    tail = _HON_TAIL.sub("", text[max(0, s-80):s])
    if _HDR_A.search(tail):
        return "A.COUNSEL", "header"
    if _HDR_R.search(tail):
        return "R.COUNSEL", "header"

    right = text[e:min(len(text), e+200)]
    left = text[max(0, s-300):s]

    right_a = list(_CUE_A.finditer(right))
    right_r = list(_CUE_R.finditer(right))
    if right_a or right_r:
        nearest_a = min((m.start() for m in right_a), default=float("inf"))
        nearest_r = min((m.start() for m in right_r), default=float("inf"))
        if nearest_a != nearest_r:
            return (
                ("A.COUNSEL", "right") if nearest_a < nearest_r
                else ("R.COUNSEL", "right")
            )

    left_a = list(_CUE_A.finditer(left))
    left_r = list(_CUE_R.finditer(left))
    if left_a or left_r:
        nearest_a = max((m.end() for m in left_a), default=-1)
        nearest_r = max((m.end() for m in left_r), default=-1)
        if nearest_a != nearest_r:
            return (
                ("A.COUNSEL", "left") if nearest_a > nearest_r
                else ("R.COUNSEL", "left")
            )

    if previous and s - previous[0] <= 200 and previous[2] != "default":
        return previous[1], "inherit"

    return "R.COUNSEL", "default"

def counsel_rule_spans(text, policy):
    output, previous = [], None
    for s, e in counsel_candidates(text):
        label, source = assign_counsel_side(text, s, e, previous)
        previous = (e, label, source)

        if source == "default":
            if policy == "drop":
                continue
            if policy == "A":
                label = "A.COUNSEL"

        output.append((s, e, label))

    return output

# ---- Judge patterns ----
_WORD = rf"(?:[A-Z]\.(?:[ \t]*[A-Z]\.)*|{_CAP})"
_NAME2 = rf"{_WORD}(?:[ \t]+{_WORD}){{0,3}}"
_TOK_RE = re.compile(_WORD)

_J_STOP = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

_J_A = re.compile(rf"(?i:\bjustice)[ \t]+({_NAME2})")
_J_B = re.compile(r",[ \t]*J{{1,2}}\b\.?")
_J_B2 = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
_J_C = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
_J_D = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
_J_E = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
_J_F = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)

JUDGE_PATTERN_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J",
    "C J.(NAME)", "D BENCH X,Y", "E J [NAME]", "F BENCH list"
]

def _trim_span(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _back_judge_name(text, end, max_tokens=4):
    window_start = max(0, end-80)
    tokens = [
        (m.start()+window_start, m.end()+window_start)
        for m in _TOK_RE.finditer(text[window_start:end])
    ]
    run = []
    for a, b in reversed(tokens):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t"):
            break
        if text[a:b].upper().strip(".") in _J_STOP:
            break
        run.insert(0, (a,b))
        if len(run) >= max_tokens:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns(text):
    result = {name: set() for name in JUDGE_PATTERN_NAMES}

    def add(name, s, e):
        span = _trim_span(text, s, e)
        if span:
            result[name].add(span)

    for m in _J_A.finditer(text):
        add(JUDGE_PATTERN_NAMES[0], m.start(1), m.end(1))

    for rx, name in [
        (_J_B, JUDGE_PATTERN_NAMES[1]),
        (_J_B2, JUDGE_PATTERN_NAMES[2])
    ]:
        for m in rx.finditer(text):
            span = _back_judge_name(text, m.start())
            if span:
                add(name, *span)

    for rx, name in [
        (_J_C, JUDGE_PATTERN_NAMES[3]),
        (_J_E, JUDGE_PATTERN_NAMES[5])
    ]:
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"}:
                if re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                    add(name, m.start(1), m.end(1))

    for m in _J_D.finditer(text):
        add(JUDGE_PATTERN_NAMES[4], m.start(1), m.end(1))

    for m in _J_F.finditer(text):
        segment, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", segment):
            k = segment.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            span = _trim_span(text, base+k, base+k+len(part2))
            if span and re.fullmatch(_NAME2, text[span[0]:span[1]]):
                result[JUDGE_PATTERN_NAMES[6]].add(span)

    return result

def resolve_nonoverlap(spans):
    result = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if result and s < result[-1][1]:
            continue
        result.append((s,e))
    return result

def judge_rule_spans(text, selected_patterns):
    spans = set()
    for name, found in judge_patterns(text).items():
        if name in selected_patterns:
            spans.update(found)
    return resolve_nonoverlap(spans)

print("Rule helpers defined.")
print("Note: counsel candidate generation is a standalone honorific-based implementation.")


In [ ]:

# CS781 — Select rule parameters on fold 1; evaluate once on fold 2
# This is a new candidate pipeline, not a claim to reproduce the old 0.299526 score.

# Choose the counsel fallback policy using fold 1 only.
def counsel_rule_metrics(texts, gold, policy):
    result = {}
    for label in ("A.COUNSEL", "R.COUNSEL"):
        g = {
            (str(d), int(e[0]), int(e[1]), label)
            for d, ents in gold.items() for e in ents
            if len(e) >= 3 and str(e[2]) == label
        }
        p = {
            (str(d), int(s), int(e), str(lab))
            for d, text in texts.items()
            for s, e, lab in counsel_rule_spans(text, policy)
            if lab == label
        }
        tp = len(g & p)
        precision = tp / len(p) if p else 0.0
        recall = tp / len(g) if g else 0.0
        result[label] = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return result

policy_scores = {
    p: counsel_rule_metrics(train_texts, train_gold, p)
    for p in ("R", "A", "drop")
}
chosen_rule_policy = max(
    policy_scores,
    key=lambda p: (
        policy_scores[p]["A.COUNSEL"] + policy_scores[p]["R.COUNSEL"],
        p == "R"
    )
)
print("Fold-1 counsel policy scores:", policy_scores)
print("Selected policy:", chosen_rule_policy)

# Select judge patterns on fold 1 only, using precision >= 0.4 and TP >= 5.
gold_judges = {
    (str(d), int(e[0]), int(e[1]))
    for d, ents in train_gold.items() for e in ents
    if len(e) >= 3 and str(e[2]) == "JUDGE"
}
judge_pattern_predictions = {n: set() for n in JUDGE_PATTERN_NAMES}

for d, text in train_texts.items():
    for name, spans in judge_patterns(text).items():
        judge_pattern_predictions[name].update((str(d), s, e) for s, e in spans)

selected_judge_patterns = []
for name in JUDGE_PATTERN_NAMES:
    pred = judge_pattern_predictions[name]
    tp = len(pred & gold_judges)
    precision = tp / len(pred) if pred else 0.0
    print(f"{name:20s} predictions={len(pred):4d} TP={tp:3d} P={precision:.3f}")
    if precision >= 0.4 and tp >= 5:
        selected_judge_patterns.append(name)

print("Selected judge patterns:", selected_judge_patterns)

# Use the current notebook's loaded threshold dictionary for this candidate.
raw_val_rules = predict_texts(champion_model, val_texts)
filtered_val_rules = apply_thr(raw_val_rules, champ_thr)

def merge_candidate_pipeline(texts, filtered_predictions):
    merged = {}
    for doc_id, text in texts.items():
        counsel = counsel_rule_spans(text, chosen_rule_policy)
        judges = [
            (s, e, "JUDGE")
            for s, e in judge_rule_spans(text, selected_judge_patterns)
        ]
        rules = counsel + judges

        kept = [
            x for x in filtered_predictions[doc_id]
            if x[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]
        kept = [
            x for x in kept
            if not any(x[0] < re_ and x[1] > rs for rs, re_, _ in rules)
        ]
        merged[doc_id] = sorted(set(kept + rules))

    return merged

candidate_val_predictions = merge_candidate_pipeline(
    val_texts, filtered_val_rules
)
candidate_val_metrics = calculate_exact_span_metrics(
    val_gold, candidate_val_predictions, ENTITY_LABELS
)

print("\nStandalone candidate pipeline validation macro-F1:",
      round(candidate_val_metrics["MACRO_F1"], 6))
print("This is NOT directly comparable to the established 0.299526 baseline")
print("until the original counsel candidate generator and thresholds are recovered.")

for label in ENTITY_LABELS:
    m = candidate_val_metrics[label]
    print(
        f"{label:12s} F1={m['f1']:.4f} P={m['precision']:.4f} "
        f"R={m['recall']:.4f} TP={m['TP']} FP={m['FP']} FN={m['FN']}"
    )

print("\nNo checkpoint or prediction file was modified.")


In [ ]:

# CS781 — RESP/WIT role and document-context audit
# Read-only diagnostic. No training, checkpoint changes, or file writes.

import re
from collections import Counter, defaultdict

assert "train_texts" in globals() and "train_gold" in globals()
assert "val_texts" in globals() and "val_gold" in globals()

ROLE_CUES = {
    "RESP": re.compile(
        r"\b(respondent|defendant|accused|opposite party|"
        r"non[- ]applicant|contesting party)\b", re.I
    ),
    "WIT": re.compile(
        r"\b(witness|PW[- ]?\d+|DW[- ]?\d+|CW[- ]?\d+|"
        r"deposed|testified|examined as|cross[- ]examined|"
        r"statement of|evidence of)\b", re.I
    ),
    "PETITIONER_OR_APPELLANT": re.compile(
        r"\b(petitioner|appellant|applicant|complainant|plaintiff)\b", re.I
    ),
    "COUNSEL": re.compile(
        r"\b(counsel|advocate|appeared for|on behalf of|"
        r"learned counsel|represented by)\b", re.I
    ),
}

WITNESS_ID = re.compile(
    r"\b(?:PW|DW|CW|P\.W|D\.W|C\.W)\.?\s*[-.]?\s*\d+[A-Z]?\b",
    re.I
)

def _gold_mentions(texts, gold, label):
    rows = []
    for doc_id, ents in gold.items():
        doc_id = str(doc_id)
        text = texts[doc_id]
        for ent in ents:
            if len(ent) < 3 or str(ent[2]) != label:
                continue
            s, e = int(ent[0]), int(ent[1])
            if not (0 <= s < e <= len(text)):
                continue
            rows.append({
                "doc": doc_id,
                "start": s,
                "end": e,
                "surface": text[s:e],
                "text": text,
            })
    return rows

def _cue_counts(row, radius=300):
    text, s, e = row["text"], row["start"], row["end"]
    lo, hi = max(0, s-radius), min(len(text), e+radius)
    context = text[lo:hi]
    return {
        name: list(rx.finditer(context))
        for name, rx in ROLE_CUES.items()
    }, context

def _show_context(row, radius=220):
    text, s, e = row["text"], row["start"], row["end"]
    left = text[max(0, s-radius):s]
    mention = text[s:e]
    right = text[e:min(len(text), e+radius)]
    return f"...{left} [[{mention}]] {right}..."

def audit_split(split_name, texts, gold, max_examples=8):
    print("\n" + "="*88)
    print(f"{split_name.upper()} — RESP/WIT CONTEXT AUDIT")
    print("="*88)

    for label in ("RESP", "WIT"):
        rows = _gold_mentions(texts, gold, label)
        doc_counts = Counter(r["doc"] for r in rows)
        surface_counts = Counter(r["surface"].strip() for r in rows)
        cue_doc_counts = Counter()
        cue_span_counts = Counter()
        witness_id_mentions = 0

        for row in rows:
            cue_matches, _ = _cue_counts(row)
            present = [name for name, matches in cue_matches.items() if matches]
            for cue in present:
                cue_span_counts[cue] += 1
                cue_doc_counts[cue] += 1
            if WITNESS_ID.search(
                row["text"][max(0, row["start"]-300):
                            min(len(row["text"]), row["end"]+300)]
            ):
                witness_id_mentions += 1

        total = len(rows)
        print(f"\n--- {label} ---")
        print(f"Gold spans: {total}; documents containing label: {len(doc_counts)}")
        print("Most frequent annotated surfaces:", surface_counts.most_common(12))
        print("Most concentrated documents:", doc_counts.most_common(8))
        if total:
            print("Largest-document share:",
                  round(max(doc_counts.values()) / total, 3))
        print("Gold spans with each cue within ±300 chars:")
        for cue in ROLE_CUES:
            print(f"  {cue:24s}: {cue_span_counts[cue]:4d}/{total}")
        if label == "WIT":
            print(f"Gold WIT spans near a PW/DW/CW-style identifier: "
                  f"{witness_id_mentions}/{total}")

        print(f"\nRepresentative {label} annotations:")
        # Show a mix of frequent and less frequent surfaces.
        chosen, seen = [], set()
        for row in rows:
            surface = row["surface"].strip().casefold()
            if surface not in seen:
                chosen.append(row)
                seen.add(surface)
            if len(chosen) >= max_examples:
                break
        for i, row in enumerate(chosen, 1):
            matches, _ = _cue_counts(row)
            nearby = [k for k, v in matches.items() if v]
            print(f"\n{i}. doc={row['doc']} span=({row['start']},{row['end']})")
            print("   Surface:", repr(row["surface"]))
            print("   Nearby cues:", nearby or "none detected")
            print("   Context:", _show_context(row))

# Run on training and validation, never test.
audit_split("train", train_texts, train_gold)
audit_split("validation", val_texts, val_gold)

# Cross-document surface overlap: descriptive only, not proof of identity.
print("\n" + "="*88)
print("TRAIN ↔ VALIDATION SURFACE OVERLAP")
print("="*88)

for label in ("RESP", "WIT"):
    train_surfaces = {
        r["surface"].strip().casefold()
        for r in _gold_mentions(train_texts, train_gold, label)
    }
    val_surfaces = {
        r["surface"].strip().casefold()
        for r in _gold_mentions(val_texts, val_gold, label)
    }
    overlap = train_surfaces & val_surfaces
    print(
        f"{label}: train unique surfaces={len(train_surfaces)}, "
        f"validation unique surfaces={len(val_surfaces)}, "
        f"overlap={len(overlap)}"
    )
    print("  Example shared surfaces:", sorted(overlap)[:20])

print("\nInterpretation:")
print("- Cue presence measures context availability, not extraction accuracy.")
print("- A nearby role cue does not prove it refers to the annotated person.")
print("- Shared name strings do not prove the same real-world person.")
print("- Use these examples to establish annotation conventions before writing rules.")
print("No model weights or files were changed.")


In [ ]:

# CS781: candidate rules for RESP headers and explicit witness identifiers.
# Validation is for evaluation only. No model or file is modified.

import re
from collections import Counter

RESP_LABEL = "RESP"
WIT_LABEL = "WIT"

# Matches common Indian judgment header styles:
# "... Versus NAME ... Respondent(s)"
# "RESPONDENT: NAME"
# "PETITIONER: NAME RESPONDENT: NAME"
RESP_MARKER = re.compile(
    r"\bRESPONDENTS?\b\s*[\(\[]?\s*[Ss]?\s*[\)\]]?\s*[:.]?",
    re.I
)
VERSUS_MARKER = re.compile(r"\b(?:VERSUS|VS\.?|V\.?)\b", re.I)

# Witness / listed-witness identifiers, including LW.
WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b",
    re.I
)

# Conservative name candidate: 1–5 capitalized name tokens, optionally
# including initials and common prefixes/titles.
NAME_TOKEN = (
    r"(?:[A-Z][A-Za-z.'’\-]*|"
    r"[A-Z]\.)"
)
NAME_CANDIDATE = re.compile(
    rf"(?<![\w])(?:Dr\.?\s+|Shri\s+|Smt\.?\s+|Mr\.?\s+|Mrs\.?\s+)?"
    rf"{NAME_TOKEN}(?:\s+{NAME_TOKEN}){{0,4}}"
)

def _add_candidate(out, text, start, end, label):
    # Trim whitespace and common punctuation without losing source offsets.
    while start < end and (text[start].isspace() or text[start] in " \t:;,."):
        start += 1
    while end > start and (text[end-1].isspace() or text[end-1] in " \t:;,."):
        end -= 1
    if start < end:
        out.add((start, end, label))

def resp_header_candidates(text, header_limit=2500):
    """Candidate spans from respondent header regions only."""
    out = set()
    header = text[:min(len(text), header_limit)]

    # Pattern A: names in the text between Versus and the Respondent marker.
    for vm in VERSUS_MARKER.finditer(header):
        rm = RESP_MARKER.search(header, vm.end())
        if rm is None:
            continue
        # Avoid grabbing a large body section if no nearby respondent label.
        if rm.start() - vm.end() > 700:
            continue

        segment_start, segment_end = vm.end(), rm.start()
        segment = header[segment_start:segment_end]

        # Split on common party separators, preserving absolute offsets.
        chunks = re.split(r"\s+(?:AND|&)\s+|[,;]\s*(?=\d|\b[A-Z])",
                          segment, flags=re.I)
        cursor = 0
        for chunk in chunks:
            local = segment.find(chunk, cursor)
            if local < 0:
                continue
            cursor = local + len(chunk)
            # Only keep name-like chunks, not legal role labels.
            for match in NAME_CANDIDATE.finditer(chunk):
                candidate = match.group(0).strip()
                if candidate.casefold() in {
                    "versus", "appellant", "appellants", "respondent",
                    "respondents", "petitioner", "petitioners"
                }:
                    continue
                s = segment_start + local + match.start()
                e = segment_start + local + match.end()
                _add_candidate(out, text, s, e, RESP_LABEL)

    # Pattern B: explicit "RESPONDENT: NAME" style metadata.
    for rm in re.finditer(r"\bRESPONDENT\b\s*:\s*", header, re.I):
        start = rm.end()
        end = min(len(header), start + 180)
        segment = header[start:end]
        # Stop at the next obvious metadata field or judgment heading.
        stop = re.search(
            r"\b(?:DATE OF JUDGMENT|BENCH|JUDGMENT|J U D G M E N T)\b",
            segment, re.I
        )
        if stop:
            segment = segment[:stop.start()]
        for match in NAME_CANDIDATE.finditer(segment):
            candidate = match.group(0).strip()
            if candidate.casefold() in {"and", "or", "ors", "the"}:
                continue
            _add_candidate(out, text, start + match.start(),
                           start + match.end(), RESP_LABEL)

    return out

def witness_id_candidates(text):
    """Candidate person spans immediately adjacent to witness identifiers."""
    out = set()

    for wm in WIT_ID.finditer(text):
        # Most common form: "Jagdish (PW-5)" or "Dr. X-PW2".
        left_start = max(0, wm.start() - 100)
        left = text[left_start:wm.start()]

        # Restrict to text after a sentence/punctuation boundary when possible.
        boundary_positions = [
            p for p in (left.rfind("."), left.rfind(";"),
                        left.rfind(":"), left.rfind("\n"))
            if p >= 0
        ]
        if boundary_positions:
            left_start += max(boundary_positions) + 1
            left = text[left_start:wm.start()]

        matches = list(NAME_CANDIDATE.finditer(left))
        if matches:
            # Keep the closest name-like expression before the identifier.
            m = matches[-1]
            candidate = m.group(0).strip()
            if candidate.casefold() not in {
                "witness", "statement", "evidence", "examined",
                "cross examined", "and", "the"
            }:
                _add_candidate(out, text, left_start + m.start(),
                               left_start + m.end(), WIT_LABEL)

        # Also handle "PW-5: Jagdish" or "LW-3, NAME".
        right = text[wm.end():min(len(text), wm.end() + 100)]
        right_boundary = re.search(r"[.;\n]", right)
        if right_boundary:
            right = right[:right_boundary.start()]
        m = NAME_CANDIDATE.search(right)
        if m:
            _add_candidate(out, text, wm.end() + m.start(),
                           wm.end() + m.end(), WIT_LABEL)

    return out

def candidates_for_text(text):
    return resp_header_candidates(text) | witness_id_candidates(text)

val_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in val_texts.items()
}
train_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in train_texts.items()
}

print("Candidate generation complete.")
print("Validation documents:", len(val_rule_candidates))
print("Validation RESP candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == RESP_LABEL))
print("Validation WIT candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == WIT_LABEL))
print("These are candidate rules, not accepted predictions.")


In [ ]:

# Exact character-span evaluation for the two candidate rules.
# No validation-derived tuning is performed here.

def _gold_set_for_label(gold, label):
    result = set()
    for doc_id, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and str(ent[2]) == label:
                result.add((str(doc_id), int(ent[0]), int(ent[1]), label))
    return result

def _candidate_set(candidate_dict, label):
    return {
        (str(doc_id), int(s), int(e), str(lab))
        for doc_id, spans in candidate_dict.items()
        for s, e, lab in spans
        if str(lab) == label
    }

def _exact_metrics(gold_set, pred_set):
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "P": round(precision, 4),
        "R": round(recall, 4),
        "F1": round(f1, 4),
    }

for label in ("RESP", "WIT"):
    gold_set = _gold_set_for_label(val_gold, label)
    pred_set = _candidate_set(val_rule_candidates, label)
    print(f"\n{label} — validation exact-span metrics")
    print("Gold spans:", len(gold_set))
    print("Candidates:", len(pred_set))
    print(_exact_metrics(gold_set, pred_set))
    print("Example exact matches:")
    for item in sorted(gold_set & pred_set)[:10]:
        doc_id, s, e, _ = item
        print(" ", doc_id, repr(val_texts[doc_id][s:e]), (s, e))

# Combined candidate output, with the two labels kept separate.
combined_gold = (
    _gold_set_for_label(val_gold, "RESP")
    | _gold_set_for_label(val_gold, "WIT")
)
combined_pred = (
    _candidate_set(val_rule_candidates, "RESP")
    | _candidate_set(val_rule_candidates, "WIT")
)
print("\nCombined RESP + WIT candidate metrics:")
print(_exact_metrics(combined_gold, combined_pred))

# Save nothing: retain outputs in memory only.
print("\nDone. Champion checkpoint and prediction files remain untouched.")


In [ ]:

# CS781 — Restore original rules + evaluate WIT hybrid
# Validation-only. No training, checkpoint writes, or prediction-file writes.

import re
from collections import Counter

_required = [
    "train_texts", "train_gold", "val_texts", "val_gold",
    "champion_model", "champ_thr", "predict_texts", "apply_thr",
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing required objects: " + ", ".join(_missing))

# ------------------------------------------------------------
# 1. Original counsel candidate rules from the original notebook
# ------------------------------------------------------------
# Reuse the original function if it exists; otherwise restore its
# conservative section-based candidate extraction.

if "rule_spans" not in globals():
    CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
    INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
    WORD = rf"(?:{INIT}|{CAP})"
    NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"

    SECTION_RE = re.compile(
        r"\b(?:"
        r"counsel\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r"|for\s+(?:the\s+)?(?:appellant|respondent|petitioner|applicant|"
        r"complainant|accused|state|opposite\s+party)s?"
        r"|appearing\s+for"
        r"|agent\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r")\s*[:\-]?\s*",
        re.I
    )
    SECTION_STOP_RE = re.compile(
        r"\b(?:Hon['’]?ble|JUDGMENT|ORDER|CORAM|"
        r"the\s+following\s+order|judgment\s+was\s+delivered)\b",
        re.I
    )

    def rule_spans(text):
        spans = set()
        for sm in SECTION_RE.finditer(text):
            start = sm.end()
            region = text[start:min(len(text), start + 500)]
            stop = SECTION_STOP_RE.search(region)
            if stop:
                region = region[:stop.start()]
            region = re.split(
                r"\n\s*\n|(?:\s{3,})|\b(?:versus|JUDGMENT|ORDER)\b",
                region, maxsplit=1, flags=re.I
            )[0]

            for m in re.finditer(NAME2, region):
                candidate = m.group(0).strip()
                if candidate.lower() in {
                    "appellant", "respondent", "petitioner", "applicant",
                    "state", "court", "justice", "judge", "counsel",
                    "advocate", "senior counsel", "learned counsel"
                }:
                    continue
                spans.add((start + m.start(), start + m.end()))
        return spans

    print(
        "NOTE: rule_spans was absent. Restored the section-based candidate "
        "extractor from the original notebook's candidate-rule family."
    )

# ------------------------------------------------------------
# 2. Original counsel side assignment and fold-1 policy selection
# ------------------------------------------------------------
PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"
CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
COUNSEL_ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    return (
        [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
        + [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    )

def _assign_side(text, s, e, prev):
    for step in COUNSEL_ORDER:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + 200))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - 300), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, source = _assign_side(text, s, e, prev)
        prev = (e, lab, source)
        if source == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def _counsel_fold1_score(policy):
    gold = {A_: set(), R_: set()}
    pred = {A_: set(), R_: set()}
    for d, ents in train_gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in gold:
                gold[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in train_texts.items():
        for s, e, lab in counsel_spans(text, policy):
            pred[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(gold[lab] & pred[lab])
        p = tp / len(pred[lab]) if pred[lab] else 0.0
        r = tp / len(gold[lab]) if gold[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

_policy_scores = {
    p: _counsel_fold1_score(p) for p in ("R", "A", "drop")
}
chosen = max(
    _policy_scores,
    key=lambda p: (
        _policy_scores[p][A_] + _policy_scores[p][R_],
        p == "R"
    )
)
print("\nFold-1 counsel policy scores:")
for p, vals in _policy_scores.items():
    print(p, vals)
print("Selected counsel policy:", chosen)

# ------------------------------------------------------------
# 3. Original judge-pattern family and fold-1-only selection
# ------------------------------------------------------------
CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
_STOP = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def _trim_judge(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start()+ws, m.end()+ws) for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") or text[a:b].upper().strip(".") in _STOP:
            break
        run.insert(0, (a, b))
        if len(run) >= max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns(text):
    out = {n: set() for n in JUDGE_NAMES}
    def add(n, s, e):
        span = _trim_judge(text, s, e)
        if span:
            out[n].add(span)
    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))
    for rx, n in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = _back_name(text, m.start())
            if span:
                add(n, *span)
    for rx, n in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} \
                    and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                add(n, m.start(1), m.end(1))
    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))
    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            span = _trim_judge(text, base+k, base+k+len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)
    return out

def _resolve_judges(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

_gold_judge = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
_pattern_preds = {n: set() for n in JUDGE_NAMES}
for d, text in train_texts.items():
    for name, spans in judge_patterns(text).items():
        _pattern_preds[name].update((str(d), s, e) for s, e in spans)

keep = []
for name in JUDGE_NAMES:
    pred = _pattern_preds[name]
    tp = len(pred & _gold_judge)
    precision = tp / len(pred) if pred else 0.0
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("\nFold-1 selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    candidates = set()
    for name, spans in judge_patterns(text).items():
        if name in selected:
            candidates.update(spans)
    return _resolve_judges(candidates)

# ------------------------------------------------------------
# 4. Rebuild original merge behavior, evaluate baseline, then WIT
# ------------------------------------------------------------
def merge_all(texts, champ_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]
        kept = [
            span for span in champ_filtered.get(d, [])
            if span[2] not in (A_, R_, "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(span[0] < re_ and span[1] > rs
                       for rs, re_, _ in rules)
        ]
        result[d] = sorted(set(kept + rules))
    return result

print("\nRecomputing validation champion predictions...")
_raw = predict_texts(champion_model, val_texts)
_filtered = apply_thr(_raw, champ_thr)
_original_baseline = merge_all(val_texts, _filtered, keep)

_baseline_metrics = calculate_exact_span_metrics(
    val_gold, _original_baseline, ENTITY_LABELS
)
print("Reconstructed baseline macro-F1:",
      round(_baseline_metrics["MACRO_F1"], 6))
print("Reference established baseline: 0.299526")
if abs(_baseline_metrics["MACRO_F1"] - 0.29952575561015876) > 0.002:
    print("WARNING: reconstructed baseline differs from reference; "
          "do not treat hybrid comparison as apples-to-apples.")

# Add WIT rule candidates only when they do not overlap existing predictions.
_wit_hybrid = {d: set(map(tuple, spans)) for d, spans in _original_baseline.items()}
_added, _blocked = 0, 0

for d, candidates in val_rule_candidates.items():
    d = str(d)
    _wit_hybrid.setdefault(d, set())
    for s, e, lab in candidates:
        if str(lab) != "WIT":
            continue
        s, e = int(s), int(e)
        if any(max(s, ps) < min(e, pe) for ps, pe, _ in _wit_hybrid[d]):
            _blocked += 1
            continue
        _wit_hybrid[d].add((s, e, "WIT"))
        _added += 1

_wit_hybrid = {d: sorted(spans) for d, spans in _wit_hybrid.items()}
_wit_metrics = calculate_exact_span_metrics(
    val_gold, _wit_hybrid, ENTITY_LABELS
)

print("\n=== FULL PIPELINE COMPARISON ===")
print(f"Baseline macro-F1:   {_baseline_metrics['MACRO_F1']:.6f}")
print(f"WIT hybrid macro-F1: {_wit_metrics['MACRO_F1']:.6f}")
print(f"Delta:               "
      f"{_wit_metrics['MACRO_F1'] - _baseline_metrics['MACRO_F1']:+.6f}")
print("WIT candidates added:", _added)
print("Candidates blocked by overlap:", _blocked)

print("\nPer-label F1:")
for lab in ENTITY_LABELS:
    b = _baseline_metrics[lab]
    h = _wit_metrics[lab]
    print(
        f"{lab:12s} {b['f1']:.4f} -> {h['f1']:.4f} "
        f"(TP {b['TP']} -> {h['TP']}, FP {b['FP']} -> {h['FP']})"
    )

# Keep only in memory.
witness_hybrid_val_predictions = _wit_hybrid
print("\nRead-only evaluation complete; no files were written.")


In [ ]:

# CS781 — Original honorific-based counsel rules + V5 side assignment
# Restored from cs781_champion_plus_rules_run_cells.ipynb
# Policy is selected on fold 1 only.

import re

HON_RE = re.compile(
    r"\b(?:Mr|Ms|Mrs|Shri|Sri|Smt)\b\.?[ \t]*(?=[A-Z])"
)
NAME_TOK = re.compile(r"[A-Z][A-Za-z\.']*")
WS = re.compile(r"[ \t]+")
SIDE_A = re.compile(
    r"petitioner|appellant|applicant|plaintiff|complainant|prosecution", re.I
)
SIDE_R = re.compile(r"respondent|defendant|accused", re.I)

def rule_spans(text):
    out = []
    for m in HON_RE.finditer(text):
        pos = m.end()
        if text[pos:pos + 7] == "Justice":
            continue
        p, name_end = pos, None
        while True:
            t = NAME_TOK.match(text, p)
            if not t:
                break
            name_end = t.end()
            w = WS.match(text, name_end)
            if not w or not text[w.end():w.end() + 1].isupper():
                break
            p = w.end()
        if name_end is None:
            continue
        e = name_end
        while e > pos and text[e - 1] in ".,;:'":
            e -= 1
        if e > pos:
            out.append((pos, e))
    return out

PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}
    P = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in G:
                G[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy):
            P[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab])
        p = tp / len(P[lab]) if P[lab] else 0.0
        r = tp / len(G[lab]) if G[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

fold1_policy_scores = {
    policy: counsel_f1(train_texts, train_gold, policy)
    for policy in ("R", "A", "drop")
}
chosen = max(
    fold1_policy_scores,
    key=lambda p: (
        fold1_policy_scores[p][A_] + fold1_policy_scores[p][R_],
        p == "R"
    )
)

print("Fold-1-only counsel policy selection:")
for policy, vals in fold1_policy_scores.items():
    print(
        f"{policy:5s} A={vals[A_]:.4f} R={vals[R_]:.4f} "
        f"mean={(vals[A_] + vals[R_])/2:.4f}"
    )
print("Selected policy:", chosen)
print("Counsel rule functions restored.")


In [ ]:

# CS781 — Original fold-1 judge-pattern selection + verified WIT hybrid
# No model training and no checkpoint/prediction files are written.

import re

CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
STOP_WORDS = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [
        (m.start() + ws, m.end() + ws)
        for m in TOK_RE.finditer(text[ws:end])
    ]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t"):
            break
        if text[a:b].upper().strip(".") in STOP_WORDS:
            break
        run.insert(0, (a, b))
        if len(run) == max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns2(text):
    out = {name: set() for name in JUDGE_NAMES}

    def add(name, s, e):
        span = trim(text, s, e)
        if span:
            out[name].add(span)

    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))

    for rx, name in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = back_name(text, m.start())
            if span:
                add(name, *span)

    for rx, name in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if (
                inner and inner[0].isupper()
                and inner.upper() not in {"J", "JJ"}
                and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner)
            ):
                add(name, m.start(1), m.end(1))

    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))

    for m in F_RE.finditer(text):
        segment, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", segment):
            k = segment.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(
                r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part
            )
            span = trim(text, base + k, base + k + len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)

    return out

def resolve_judge_overlaps(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

# Select patterns on fold 1 only.
gold_judge_train = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents
    if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
pattern_preds = {name: set() for name in JUDGE_NAMES}

for d, text in train_texts.items():
    for name, spans in judge_patterns2(text).items():
        pattern_preds[name].update(
            (str(d), s, e) for s, e in spans
        )

keep = []
print("Fold-1-only judge-pattern selection:")
for name in JUDGE_NAMES:
    pred = pattern_preds[name]
    tp = len(gold_judge_train & pred)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold_judge_train) if gold_judge_train else 0.0
    print(
        f"{name:26s} predicted={len(pred):4d} TP={tp:3d} "
        f"P={precision:.3f} R={recall:.3f}"
    )
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("Selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    spans = set()
    for name, candidates in judge_patterns2(text).items():
        if name in selected:
            spans.update(candidates)
    return resolve_judge_overlaps(spans)

# Use the established champion thresholds, not finalv1's altered champ_thr.
ESTABLISHED_THR = {
    "APP": 0.15,
    "RESP": 0.45,
    "A.COUNSEL": 0.45,
    "R.COUNSEL": 0.45,
    "JUDGE": 0.25,
    "WIT": 0.45,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.25,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Verify the inference helpers and candidate rules exist.
for name in (
    "champion_model", "predict_texts", "apply_thr",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates", "val_texts", "val_gold"
):
    if name not in globals():
        raise RuntimeError(f"Missing required object: {name}")

# Build baseline: thresholded champion + original counsel + selected judges.
raw_val_original = predict_texts(champion_model, val_texts)
filtered_val_original = apply_thr(raw_val_original, ESTABLISHED_THR)

def merge_all_original(texts, champion_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen))
        rules += [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]

        kept = [
            span for span in champion_filtered.get(d, [])
            if span[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(
                span[0] < rule_e and span[1] > rule_s
                for rule_s, rule_e, _ in rules
            )
        ]
        result[d] = sorted(set(kept + rules))
    return result

baseline_original = merge_all_original(
    val_texts, filtered_val_original, keep
)
baseline_metrics = calculate_exact_span_metrics(
    val_gold, baseline_original, ENTITY_LABELS
)

print("\n=== VERIFIED BASELINE CHECK ===")
print("Reconstructed macro-F1:", round(baseline_metrics["MACRO_F1"], 6))
print("Reference macro-F1:     0.299526")
baseline_matches_reference = (
    abs(baseline_metrics["MACRO_F1"] - 0.29952575561015876) <= 0.002
)
print("Baseline matches reference:", baseline_matches_reference)

# Add only WIT candidates not overlapping an existing baseline prediction.
wit_hybrid = {
    str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
    for d, spans in baseline_original.items()
}
added, blocked = 0, 0

for d, candidates in val_rule_candidates.items():
    d = str(d)
    wit_hybrid.setdefault(d, set())
    for s, e, lab in candidates:
        if str(lab) != "WIT":
            continue
        s, e = int(s), int(e)
        if any(max(s, ps) < min(e, pe) for ps, pe, _ in wit_hybrid[d]):
            blocked += 1
            continue
        wit_hybrid[d].add((s, e, "WIT"))
        added += 1

wit_hybrid = {d: sorted(spans) for d, spans in wit_hybrid.items()}
wit_metrics = calculate_exact_span_metrics(
    val_gold, wit_hybrid, ENTITY_LABELS
)

print("\n=== FULL PIPELINE COMPARISON ===")
print(f"Baseline macro-F1:   {baseline_metrics['MACRO_F1']:.6f}")
print(f"WIT hybrid macro-F1: {wit_metrics['MACRO_F1']:.6f}")
print(f"Delta:               "
      f"{wit_metrics['MACRO_F1'] - baseline_metrics['MACRO_F1']:+.6f}")
print("WIT candidates added:", added)
print("Candidates blocked by overlap:", blocked)

print("\nPer-label comparison:")
for lab in ENTITY_LABELS:
    b, h = baseline_metrics[lab], wit_metrics[lab]
    print(
        f"{lab:12s} F1 {b['f1']:.4f} -> {h['f1']:.4f} | "
        f"TP {b['TP']} -> {h['TP']} | FP {b['FP']} -> {h['FP']}"
    )

# Keep the candidate result in memory only.
witness_hybrid_val_predictions = wit_hybrid

if not baseline_matches_reference:
    print(
        "\nSTOP: baseline differs from 0.299526. Do not accept or reject "
        "the WIT hybrid yet; the original baseline configuration still "
        "does not match exactly."
    )
else:
    print(
        "\nBaseline check passed. Compare the WIT hybrid delta above; "
        "nothing was saved or overwritten."
    )


In [ ]:

# CS781 — Inspect WIT false positives from the verified hybrid

assert "witness_hybrid_val_predictions" in globals()
assert "val_gold" in globals()
assert "val_texts" in globals()

def _span_set_by_label(predictions, label):
    return {
        (str(doc_id), int(s), int(e), str(lab))
        for doc_id, spans in predictions.items()
        for s, e, lab in spans
        if str(lab) == label
    }

def _gold_label_set(gold, label):
    return {
        (str(doc_id), int(ent[0]), int(ent[1]), str(ent[2]))
        for doc_id, ents in gold.items()
        for ent in ents
        if len(ent) >= 3 and str(ent[2]) == label
    }

wit_pred = _span_set_by_label(witness_hybrid_val_predictions, "WIT")
wit_gold = _gold_label_set(val_gold, "WIT")
wit_fp = sorted(wit_pred - wit_gold)
wit_fn = sorted(wit_gold - wit_pred)

print("WIT predicted:", len(wit_pred))
print("WIT gold:", len(wit_gold))
print("WIT TP:", len(wit_pred & wit_gold))
print("WIT FP:", len(wit_fp))
print("WIT FN:", len(wit_fn))

ID_RE = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b", re.I
)

def _nearby_gold_wit(doc_id, s, e, radius=100):
    return [
        (gs, ge, val_texts[doc_id][gs:ge])
        for d, gs, ge, lab in wit_gold
        if d == doc_id and max(s, gs) < min(e, ge)
    ]

print("\n=== WIT FALSE POSITIVES ===")
for i, (doc_id, s, e, label) in enumerate(wit_fp, 1):
    text = val_texts[doc_id]
    lo, hi = max(0, s-180), min(len(text), e+180)
    context = text[lo:hi].replace("\n", " ")
    nearby_ids = [m.group(0) for m in ID_RE.finditer(
        text[max(0, s-250):min(len(text), e+250)]
    )]
    overlaps = _nearby_gold_wit(doc_id, s, e)

    print(f"\n{i}. doc={doc_id}, span=({s},{e})")
    print("   Predicted text:", repr(text[s:e]))
    print("   Nearby witness IDs:", nearby_ids or "none")
    print("   Overlapping gold WIT:", overlaps or "none")
    print("   Context:", context)

print("\n=== WIT FALSE NEGATIVES ===")
for i, (doc_id, s, e, label) in enumerate(wit_fn, 1):
    text = val_texts[doc_id]
    lo, hi = max(0, s-150), min(len(text), e+150)
    context = text[lo:hi].replace("\n", " ")
    print(f"\n{i}. doc={doc_id}, gold=({s},{e})")
    print("   Gold text:", repr(text[s:e]))
    print("   Context:", context)


In [ ]:

# CS781 — Compare WIT filters against the verified baseline
# No test data, training, or file writes.

assert "baseline_original" in globals()
assert "calculate_exact_span_metrics" in globals()
assert "ENTITY_LABELS" in globals()
assert "val_rule_candidates" in globals()

_wit_id_pattern = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b", re.I
)

def _has_id_near(text, s, e, radius):
    region = text[max(0, s-radius):min(len(text), e+radius)]
    return bool(_wit_id_pattern.search(region))

def _make_wit_hybrid(mode):
    # Start from a fresh copy of the verified baseline.
    result = {
        str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
        for d, spans in baseline_original.items()
    }
    added = 0
    blocked = 0
    rejected = 0

    for d, candidates in val_rule_candidates.items():
        d = str(d)
        text = val_texts[d]
        result.setdefault(d, set())

        for s, e, lab in candidates:
            if str(lab) != "WIT":
                continue
            s, e = int(s), int(e)

            if mode == "has_identifier" and not _has_id_near(text, s, e, 150):
                rejected += 1
                continue
            if mode == "tight_identifier" and not _has_id_near(text, s, e, 60):
                rejected += 1
                continue

            if any(
                max(s, ps) < min(e, pe)
                for ps, pe, _ in result[d]
            ):
                blocked += 1
                continue

            result[d].add((s, e, "WIT"))
            added += 1

    return {d: sorted(spans) for d, spans in result.items()}, added, blocked, rejected

def _metrics_for_candidate(predictions):
    return calculate_exact_span_metrics(
        val_gold, predictions, ENTITY_LABELS
    )

candidate_results = {}
for mode in ("all", "has_identifier", "tight_identifier"):
    pred, added, blocked, rejected = _make_wit_hybrid(mode)
    metrics = _metrics_for_candidate(pred)
    wit = metrics["WIT"]
    candidate_results[mode] = (pred, metrics)

    print(f"\n=== {mode} ===")
    print("Macro-F1:", round(metrics["MACRO_F1"], 6))
    print("Delta vs verified baseline:",
          round(metrics["MACRO_F1"] - baseline_metrics["MACRO_F1"], 6))
    print("WIT P/R/F1:",
          round(wit["precision"], 4),
          round(wit["recall"], 4),
          round(wit["f1"], 4))
    print("WIT TP/FP/FN:", wit["TP"], wit["FP"], wit["FN"])
    print("Added / blocked / rejected:", added, blocked, rejected)

print("\nBaseline macro-F1:", round(baseline_metrics["MACRO_F1"], 6))


In [ ]:

# CS781 — Select validation candidate in memory only
# Do not use this selection to claim test performance.

_order = {"all": 0, "has_identifier": 1, "tight_identifier": 2}
_best_mode = max(
    candidate_results,
    key=lambda mode: (
        candidate_results[mode][1]["MACRO_F1"],
        -_order[mode]
    )
)

best_wit_val_predictions, best_wit_val_metrics = candidate_results[_best_mode]

print("Selected validation candidate:", _best_mode)
print("Verified baseline macro-F1:",
      round(baseline_metrics["MACRO_F1"], 6))
print("Selected candidate macro-F1:",
      round(best_wit_val_metrics["MACRO_F1"], 6))
print("Gain:",
      round(
          best_wit_val_metrics["MACRO_F1"] - baseline_metrics["MACRO_F1"],
          6
      ))
print("WIT metrics:", best_wit_val_metrics["WIT"])
print("No files were written; champion artifacts are untouched.")


In [ ]:

# CS781 — Conservative WIT candidate cleanup
# Uses the verified baseline and validation labels only.

import re

assert "baseline_original" in globals()
assert "val_rule_candidates" in globals()
assert "val_texts" in globals()
assert "val_gold" in globals()
assert "calculate_exact_span_metrics" in globals()

_WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b", re.I
)

# Terms seen in the error audit that are not person-name spans in context.
_NON_PERSON = {
    "it", "both", "pw", "dw", "cw", "lw", "a", "an", "the",
    "dr", "court", "superintendent", "ex-sarpanch",
    "witness", "evidence", "statement", "respondent", "appellant",
    "accused", "petitioner", "deceased", "complainant",
}

def _near_identifier(text, s, e, radius=60):
    region = text[max(0, s-radius):min(len(text), e+radius)]
    return bool(_WIT_ID.search(region))

def _candidate_surface(text, s, e):
    return text[s:e].strip(" \t\r\n,.;:()[]{}-–—")

def _valid_person_surface(surface, mode):
    normalized = surface.casefold().strip()
    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", surface)

    if normalized in _NON_PERSON:
        return False

    # Reject fragments that are just witness-ID tokens.
    if re.fullmatch(r"(?:P|D|C|L)?W\d*", normalized.replace("-", "")):
        return False

    if mode in ("non_person", "name_shape", "name_shape_plus_context"):
        # Require at least one alphabetic word of reasonable length.
        if not words or max(map(len, words)) < 3:
            return False

    if mode in ("name_shape", "name_shape_plus_context"):
        # A single token can be a legitimate name, so do not reject it
        # merely for having one word.
        if len(surface.strip()) < 4:
            return False

    return True

def _run_wit_filter(mode):
    pred = {
        str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
        for d, spans in baseline_original.items()
    }
    added = 0
    rejected = 0
    blocked = 0

    for d, candidates in val_rule_candidates.items():
        d = str(d)
        text = val_texts[d]
        pred.setdefault(d, set())

        for s, e, label in candidates:
            if str(label) != "WIT":
                continue
            s, e = int(s), int(e)
            surface = _candidate_surface(text, s, e)

            # All variants start from the same 60-character identifier rule.
            if not _near_identifier(text, s, e, radius=60):
                rejected += 1
                continue

            if mode != "identifier_only":
                if not _valid_person_surface(surface, mode):
                    rejected += 1
                    continue

            if mode == "name_shape_plus_context":
                # Require an explicit identifier very close to the candidate.
                # This is deliberately conservative.
                if not _near_identifier(text, s, e, radius=30):
                    rejected += 1
                    continue

            if any(
                max(s, ps) < min(e, pe)
                for ps, pe, _ in pred[d]
            ):
                blocked += 1
                continue

            pred[d].add((s, e, "WIT"))
            added += 1

    pred = {d: sorted(spans) for d, spans in pred.items()}
    metrics = calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)
    return pred, metrics, added, rejected, blocked

_wit_filter_results = {}
for _mode in (
    "identifier_only",
    "non_person",
    "name_shape",
    "name_shape_plus_context",
):
    _pred, _metrics, _added, _rejected, _blocked = _run_wit_filter(_mode)
    _wit_filter_results[_mode] = (_pred, _metrics)

    _w = _metrics["WIT"]
    print(f"\n=== {_mode} ===")
    print("Macro-F1:", round(_metrics["MACRO_F1"], 6))
    print("Delta vs verified baseline:",
          round(_metrics["MACRO_F1"] - baseline_metrics["MACRO_F1"], 6))
    print(
        f"WIT P={_w['precision']:.4f} R={_w['recall']:.4f} "
        f"F1={_w['f1']:.4f} TP={_w['TP']} FP={_w['FP']} FN={_w['FN']}"
    )
    print("Added:", _added, "Rejected:", _rejected, "Blocked:", _blocked)

print("\nBaseline macro-F1:", round(baseline_metrics["MACRO_F1"], 6))
print("No files were written.")


In [ ]:

# CS781 — Inspect the trade-off between name_shape and name_shape_plus_context
# Validation only; read-only.

_name_pred, _name_metrics = _wit_filter_results["name_shape"]
_ctx_pred, _ctx_metrics = _wit_filter_results["name_shape_plus_context"]

_name_wit = _span_set_by_label(_name_pred, "WIT")
_ctx_wit = _span_set_by_label(_ctx_pred, "WIT")
_gold_wit = _gold_label_set(val_gold, "WIT")

removed_by_context = sorted(_name_wit - _ctx_wit)
print("Candidates removed by context filter:", len(removed_by_context))
print("Removed gold spans:", len(set(removed_by_context) & _gold_wit))
print("Removed false positives:", len(set(removed_by_context) - _gold_wit))

for i, (doc_id, s, e, label) in enumerate(removed_by_context, 1):
    text = val_texts[doc_id]
    lo, hi = max(0, s-180), min(len(text), e+180)
    print(f"\n{i}. doc={doc_id}, span=({s},{e})")
    print("   Text:", repr(text[s:e]))
    print("   Gold witness:", (doc_id, s, e, "WIT") in _gold_wit)
    print("   Context:", text[lo:hi].replace("\n", " "))

print("\n=== METRIC COMPARISON ===")
for mode in ("name_shape", "name_shape_plus_context"):
    _, m = _wit_filter_results[mode]
    w = m["WIT"]
    print(
        f"{mode:24s} macro={m['MACRO_F1']:.6f} "
        f"WIT P={w['precision']:.4f} R={w['recall']:.4f} "
        f"F1={w['f1']:.4f} TP={w['TP']} FP={w['FP']} FN={w['FN']}"
    )


In [ ]:

# Inspect the remaining WIT false positives for the strongest filter.
# This is a validation error analysis only; it does not modify predictions.

_ctx_pred, _ctx_metrics = _wit_filter_results["name_shape_plus_context"]
_gold_wit = _gold_label_set(val_gold, "WIT")
_ctx_wit = _span_set_by_label(_ctx_pred, "WIT")

remaining_fp = sorted(_ctx_wit - _gold_wit)
print("Remaining WIT false positives:", len(remaining_fp))

for i, (doc_id, s, e, label) in enumerate(remaining_fp, 1):
    text = val_texts[doc_id]
    lo, hi = max(0, s - 140), min(len(text), e + 140)
    print(f"\n{i}. doc={doc_id}, span=({s},{e})")
    print("   Surface:", repr(text[s:e]))
    print("   Context:", text[lo:hi].replace("\n", " "))


In [ ]:

# CS781 — Conservative WIT phrase exclusions
# Uses validation only and does not overwrite any saved predictions.

import re

_EXTRA_WIT_EXCLUSIONS = {
    "both pw",
    "the police patil",
    "sub inspector",
    "sub-inspector",
    "associate professor",
    "the fir",
    "subsequently",
    "oblique",
    "manchili",
    "palshi",
}

def _apply_extra_wit_exclusions(base_predictions):
    result = {
        doc_id: list(spans)
        for doc_id, spans in base_predictions.items()
    }

    removed = []
    for doc_id, spans in result.items():
        text = val_texts[doc_id]
        kept = []
        for span in spans:
            s, e, label = span[:3]
            surface = text[s:e].strip()

            if label == "WIT" and surface.casefold() in _EXTRA_WIT_EXCLUSIONS:
                removed.append((doc_id, s, e, surface))
            else:
                kept.append(span)

        result[doc_id] = kept

    return result, removed

_context_base, _ = _wit_filter_results["name_shape_plus_context"]
_extra_pred, _extra_removed = _apply_extra_wit_exclusions(_context_base)
_extra_metrics = calculate_exact_span_metrics(
    val_gold, _extra_pred, ENTITY_LABELS
)

print("Context-filter macro-F1:", 
      _wit_filter_results["name_shape_plus_context"][1]["MACRO_F1"])
print("With extra exclusions:", _extra_metrics["MACRO_F1"])
print("Delta:", _extra_metrics["MACRO_F1"] -
      _wit_filter_results["name_shape_plus_context"][1]["MACRO_F1"])
print("Candidates removed:", len(_extra_removed))
for item in _extra_removed:
    print(item)

w = _extra_metrics["WIT"]
print(
    f"WIT P={w['precision']:.4f} R={w['recall']:.4f} "
    f"F1={w['f1']:.4f} TP={w['TP']} FP={w['FP']} FN={w['FN']}"
)


In [ ]:

# CS781 — Leave-one-exclusion-out ablation on validation
# Does not modify the champion or write prediction files.

_context_base, _ = _wit_filter_results["name_shape_plus_context"]

_exclusions = sorted(_EXTRA_WIT_EXCLUSIONS)

def _score_wit_exclusions(exclusions):
    exclusions = {x.casefold() for x in exclusions}
    pred = {
        doc_id: list(spans)
        for doc_id, spans in _context_base.items()
    }

    for doc_id, spans in pred.items():
        text = val_texts[doc_id]
        pred[doc_id] = [
            span for span in spans
            if not (
                span[2] == "WIT"
                and text[span[0]:span[1]].strip().casefold() in exclusions
            )
        ]

    return calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)

print("Full exclusion set:")
_full = _score_wit_exclusions(_exclusions)
print(f"Macro-F1={_full['MACRO_F1']:.6f}")

print("\nLeave one exclusion out:")
for excluded in _exclusions:
    remaining = [x for x in _exclusions if x != excluded]
    m = _score_wit_exclusions(remaining)
    print(
        f"omit={excluded!r:24s} "
        f"macro={m['MACRO_F1']:.6f} "
        f"delta_vs_full={m['MACRO_F1'] - _full['MACRO_F1']:+.6f}"
    )


In [ ]:

# CS781 — Final experimental WIT rule
# Validation/test inference only. Does not modify champion files.

FINAL_WIT_EXCLUSIONS = {
    "associate professor",
    "both pw",
    "manchili",
    "oblique",
    "subsequently",
    "the police patil",
}

_context_base, _ = _wit_filter_results["name_shape_plus_context"]

def apply_final_wit_rule(base_predictions, texts):
    result = {
        doc_id: list(spans)
        for doc_id, spans in base_predictions.items()
    }
    removed = []

    for doc_id, spans in result.items():
        text = texts[doc_id]
        kept = []

        for span in spans:
            s, e, label = span[:3]
            surface = text[s:e].strip()

            if (
                label == "WIT"
                and surface.casefold() in FINAL_WIT_EXCLUSIONS
            ):
                removed.append((doc_id, s, e, surface))
            else:
                kept.append(span)

        result[doc_id] = kept

    return result, removed


final_wit_val_predictions, final_wit_removed = apply_final_wit_rule(
    _context_base, val_texts
)

final_wit_val_metrics = calculate_exact_span_metrics(
    val_gold, final_wit_val_predictions, ENTITY_LABELS
)

baseline_f1 = baseline_metrics["MACRO_F1"]
final_f1 = final_wit_val_metrics["MACRO_F1"]
w = final_wit_val_metrics["WIT"]

print(f"Verified baseline macro-F1: {baseline_f1:.6f}")
print(f"Final WIT-rule macro-F1:    {final_f1:.6f}")
print(f"Delta:                      {final_f1 - baseline_f1:+.6f}")
print(f"Removed candidates:         {len(final_wit_removed)}")
print(
    f"WIT P={w['precision']:.4f} R={w['recall']:.4f} "
    f"F1={w['f1']:.4f} TP={w['TP']} FP={w['FP']} FN={w['FN']}"
)


In [ ]:

# Full per-label comparison: baseline vs final WIT rule

print(f"{'LABEL':12s} {'BASE F1':>10s} {'NEW F1':>10s} {'DELTA':>10s}")
print("-" * 46)

for label in ENTITY_LABELS:
    b = baseline_metrics[label]
    n = final_wit_val_metrics[label]
    print(
        f"{label:12s} "
        f"{b['f1']:10.4f} {n['f1']:10.4f} "
        f"{n['f1'] - b['f1']:+10.4f}"
    )

print("-" * 46)
print(f"Baseline macro-F1: {baseline_metrics['MACRO_F1']:.6f}")
print(f"New macro-F1:      {final_wit_val_metrics['MACRO_F1']:.6f}")


In [ ]:

import json
from pathlib import Path

OUT_DIR = Path("/kaggle/working")
val_output_path = OUT_DIR / "prediction_wit_context_experiment_val.json"

val_serialized = {}
for doc_id, spans in final_wit_val_predictions.items():
    val_serialized[str(doc_id)] = [
        [int(s), int(e), str(label)]
        for s, e, label, *rest in spans
    ]

with open(val_output_path, "w", encoding="utf-8") as f:
    json.dump(val_serialized, f, ensure_ascii=False)

print("Saved validation experiment:", val_output_path)
print("Documents:", len(val_serialized))
print("Champion file left untouched.")


In [ ]:

# CS781 — Final experimental test predictions
# Uses the already-loaded champion_model. No training.
# Does NOT overwrite prediction_champion_plus_rules.json.

import json
from pathlib import Path

# Verify that the notebook's existing inference setup is available.
_required = [
    "champion_model",
    "test_texts",
    "predict_texts",
    "apply_thr",
    "merge_all",
    "keep",
    "chosen",
    "champ_thr",
    "apply_final_wit_rule",
    "FINAL_WIT_EXCLUSIONS",
    "ENTITY_LABELS",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "Missing notebook definitions: " + ", ".join(_missing)
        + ". Run the earlier setup/rule cells first."
    )

# Use the existing champion model, not the undefined variable `model`.
champion_model.eval()
test_raw = predict_texts(champion_model, test_texts)

# Apply the notebook's current champion thresholds and original rule pipeline.
test_filtered = apply_thr(test_raw, champ_thr)
test_baseline = merge_all(test_texts, test_filtered, keep)

# Apply the conservative WIT context filter and six exclusions.
test_final_predictions, test_removed = apply_final_wit_rule(
    test_baseline, test_texts
)

# Validate IDs, span boundaries, and labels before saving.
assert set(test_final_predictions) == set(test_texts)

for doc_id, spans in test_final_predictions.items():
    text = test_texts[doc_id]
    for s, e, label in spans:
        assert 0 <= s < e <= len(text), (doc_id, s, e, len(text))
        assert label in ENTITY_LABELS, (doc_id, label)

# New filename: preserve the champion prediction file.
out_path = Path(
    "/kaggle/working/prediction_wit_context_experiment.json"
)
serialized = {
    str(doc_id): [
        [int(s), int(e), str(label)]
        for s, e, label in spans
    ]
    for doc_id, spans in test_final_predictions.items()
}

with out_path.open("w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False)

print("Saved:", out_path)
print("Documents:", len(serialized))
print("WIT candidates removed by exclusions:", len(test_removed))
print("Total predicted spans:", sum(map(len, serialized.values())))
print("Champion prediction file was not overwritten.")
print("No test gold labels were used.")


In [ ]:

# EXPERIMENT 1A — Baseline reproducibility + hybrid prerequisites
# Read-only: no training, no files written.

_required = [
    "champion_model", "predict_texts", "apply_thr",
    "champ_thr", "merge_all", "keep",
    "val_texts", "val_gold", "test_texts",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Run setup cells first. Missing: " + ", ".join(_missing))

champion_model.eval()
exp1_raw_val = predict_texts(champion_model, val_texts)
exp1_thr_val = apply_thr(exp1_raw_val, champ_thr)
exp1_base_val = merge_all(val_texts, exp1_thr_val, keep)
exp1_base_metrics = calculate_exact_span_metrics(
    val_gold, exp1_base_val, ENTITY_LABELS
)

print("Current threshold map:", champ_thr)
print("Fresh baseline macro-F1:", round(exp1_base_metrics["MACRO_F1"], 6))
print("Reference full champion + rules:", 0.299526)
print("WIT rule helpers available:", {
    name: name in globals()
    for name in [
        "witness_id_candidates", "NAME_CANDIDATE",
        "_wit_filter_results", "FINAL_WIT_EXCLUSIONS"
    ]
})
print("Champion weights unchanged.")


In [ ]:

# EXPERIMENT 2 — Small coordinate threshold sweep on fold 2
# Validation only. No model changes or file writes.

import itertools

exp2_labels = ["APP", "AUTH", "PREC", "CASENO", "STAT"]
exp2_grid = {
    "APP":    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    "AUTH":   [0.20, 0.25, 0.30, 0.35, 0.40, 0.45],
    "PREC":   [0.20, 0.25, 0.30, 0.35, 0.40, 0.45],
    "CASENO": [0.20, 0.25, 0.30, 0.35, 0.40, 0.45],
    "STAT":   [0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
}

exp2_thresholds = dict(champ_thr)
exp2_baseline = calculate_exact_span_metrics(
    val_gold, apply_thr(exp1_raw_val, exp2_thresholds), ENTITY_LABELS
)["MACRO_F1"]

print(f"Starting threshold-only macro-F1: {exp2_baseline:.6f}")

# Coordinate descent: tune one label at a time, repeat twice.
# Each step selects the candidate with the best overall macro-F1.
exp2_history = []

for pass_no in range(1, 3):
    print(f"\n=== Pass {pass_no} ===")
    changed = False

    for lab in exp2_labels:
        candidates = []

        for threshold in exp2_grid[lab]:
            trial_thr = dict(exp2_thresholds)
            trial_thr[lab] = threshold
            trial_pred = apply_thr(exp1_raw_val, trial_thr)
            trial_metrics = calculate_exact_span_metrics(
                val_gold, trial_pred, ENTITY_LABELS
            )
            candidates.append((
                trial_metrics["MACRO_F1"],
                threshold,
                trial_metrics[lab]["f1"],
                trial_metrics[lab]["precision"],
                trial_metrics[lab]["recall"],
            ))

        candidates.sort(reverse=True)
        best = candidates[0]
        old_thr = exp2_thresholds[lab]
        old_score = calculate_exact_span_metrics(
            val_gold, apply_thr(exp1_raw_val, exp2_thresholds), ENTITY_LABELS
        )["MACRO_F1"]

        if best[0] > old_score + 1e-10:
            exp2_thresholds[lab] = best[1]
            changed = changed or best[1] != old_thr

        exp2_history.append({
            "pass": pass_no,
            "label": lab,
            "chosen_threshold": exp2_thresholds[lab],
            "best_grid_macro": best[0],
            "label_f1_at_best": best[2],
        })
        print(
            f"{lab:8s} threshold={exp2_thresholds[lab]:.2f} "
            f"best_grid_macro={best[0]:.6f} "
            f"label_F1={best[2]:.4f}"
        )

    if not changed:
        print("No threshold changes in this pass; stopping.")
        break

exp2_thr_pred = apply_thr(exp1_raw_val, exp2_thresholds)
exp2_thr_metrics = calculate_exact_span_metrics(
    val_gold, exp2_thr_pred, ENTITY_LABELS
)

print("\n=== FINAL THRESHOLD RESULT ===")
print("Starting threshold-only macro-F1:", round(exp2_baseline, 6))
print("Tuned threshold-only macro-F1:   ", round(exp2_thr_metrics["MACRO_F1"], 6))
print("Delta:", round(exp2_thr_metrics["MACRO_F1"] - exp2_baseline, 6))
print("Candidate thresholds:", exp2_thresholds)


In [ ]:

# EXPERIMENT 3 — Error audit for weak labels
# Uses the threshold-only baseline and gold validation spans.

exp3_labels = ["APP", "AUTH", "PREC", "CASENO", "STAT"]

def exp3_span_sets(gold_by_doc, pred_by_doc, label):
    gold_set, pred_set = set(), set()

    for doc_id, ents in gold_by_doc.items():
        for ent in ents:
            if len(ent) >= 3 and str(ent[2]) == label:
                gold_set.add((str(doc_id), int(ent[0]), int(ent[1]), label))

    for doc_id, ents in pred_by_doc.items():
        for ent in ents:
            if len(ent) >= 3 and str(ent[2]) == label:
                pred_set.add((str(doc_id), int(ent[0]), int(ent[1]), label))

    return gold_set, pred_set

exp3_pred = apply_thr(exp1_raw_val, champ_thr)

for lab in exp3_labels:
    gold_set, pred_set = exp3_span_sets(val_gold, exp3_pred, lab)
    fp = sorted(pred_set - gold_set)
    fn = sorted(gold_set - pred_set)

    print("\n" + "=" * 90)
    print(f"{lab}: FP={len(fp)} | FN={len(fn)}")
    print("\nFALSE POSITIVES (up to 8)")
    for doc_id, s, e, _ in fp[:8]:
        text = val_texts[doc_id]
        print(
            f"\n{doc_id} [{s}:{e}] {text[s:e]!r}\n"
            f"  ...{text[max(0,s-120):min(len(text),e+120)].replace(chr(10),' ')}..."
        )

    print("\nFALSE NEGATIVES (up to 8)")
    for doc_id, s, e, _ in fn[:8]:
        text = val_texts[doc_id]
        print(
            f"\n{doc_id} [{s}:{e}] {text[s:e]!r}\n"
            f"  ...{text[max(0,s-120):min(len(text),e+120)].replace(chr(10),' ')}..."
        )


In [ ]:

# EXPERIMENT 4 — Gold-token probability audit
# Reuses the notebook's diag_doc_probs() and BIO label mapping.

_required = ["diag_doc_probs", "label2id", "BIO_LABELS", "champion_model"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing definitions: " + ", ".join(_missing))

from collections import Counter, defaultdict
import numpy as np

def exp4_audit(texts, gold, labels=("APP", "AUTH", "PREC", "CASENO", "STAT")):
    summary = defaultdict(lambda: {
        "gold_prob": [], "top_tags": Counter(), "tokens": 0, "target_top1": 0
    })

    for doc_id, text in texts.items():
        offsets, probs = diag_doc_probs(champion_model, text)
        if len(offsets) == 0:
            continue
        top_ids = probs.argmax(-1).tolist()

        for ent in gold[str(doc_id)]:
            if len(ent) < 3:
                continue
            s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
            if lab not in labels:
                continue

            ix = [
                i for i, (a, b) in enumerate(offsets)
                if b > a and a < e and b > s
            ]
            for j, i in enumerate(ix):
                tag = ("B-" if j == 0 else "I-") + lab
                tid = label2id[tag]
                summary[lab]["gold_prob"].append(float(probs[i, tid]))
                summary[lab]["top_tags"][BIO_LABELS[top_ids[i]]] += 1
                summary[lab]["tokens"] += 1
                summary[lab]["target_top1"] += int(top_ids[i] == tid)

    for lab in labels:
        d = summary[lab]
        mean_prob = float(np.mean(d["gold_prob"])) if d["gold_prob"] else float("nan")
        print(f"\n{lab}: gold-token count={d['tokens']}")
        print(f"Mean probability of correct BIO tag: {mean_prob:.4f}")
        print(f"Correct tag top-1: {d['target_top1']}/{d['tokens']}")
        print("Most common competing tags:", d["top_tags"].most_common(8))

print("=== TRAIN FOLD ===")
exp4_audit(train_texts, train_gold)
print("\n=== VALIDATION FOLD ===")
exp4_audit(val_texts, val_gold)


In [ ]:

# EXPERIMENT 5 — Compare threshold candidates after the original rules
# Validation only. No files written.

# Original threshold configuration.
exp5_original = dict(champ_thr)

# Baseline after original rules.
exp5_base_filtered = apply_thr(exp1_raw_val, exp5_original)
exp5_base_merged = merge_all(val_texts, exp5_base_filtered, keep)
exp5_base_metrics = calculate_exact_span_metrics(
    val_gold, exp5_base_merged, ENTITY_LABELS
)

# Tuned threshold configuration after the same original rules.
exp5_tuned_filtered = apply_thr(exp1_raw_val, exp2_thresholds)
exp5_tuned_merged = merge_all(val_texts, exp5_tuned_filtered, keep)
exp5_tuned_metrics = calculate_exact_span_metrics(
    val_gold, exp5_tuned_merged, ENTITY_LABELS
)

print("=== FULL PIPELINE COMPARISON ===")
print(f"Original thresholds: {exp5_base_metrics['MACRO_F1']:.6f}")
print(f"Tuned thresholds:    {exp5_tuned_metrics['MACRO_F1']:.6f}")
print(f"Delta:               {exp5_tuned_metrics['MACRO_F1'] - exp5_base_metrics['MACRO_F1']:+.6f}")

print("\nPer-label F1:")
for lab in ENTITY_LABELS:
    old = exp5_base_metrics[lab]["f1"]
    new = exp5_tuned_metrics[lab]["f1"]
    print(f"{lab:12s} {old:.4f} -> {new:.4f} ({new-old:+.4f})")

if exp5_tuned_metrics["MACRO_F1"] > exp5_base_metrics["MACRO_F1"]:
    print("\nPromising on validation, but not yet proven on unseen data.")
else:
    print("\nReject this threshold candidate; it did not improve the full pipeline.")


In [ ]:

# EXPERIMENT 6 — Train vs validation distribution audit
# Read-only: no training, no prediction files written.

from collections import Counter, defaultdict
import numpy as np

def exp6_normalize_gold(gold):
    """Normalize document-level gold spans to (start, end, label)."""
    result = {}
    for doc_id, entities in gold.items():
        spans = []
        for ent in entities:
            if len(ent) >= 3:
                spans.append((int(ent[0]), int(ent[1]), str(ent[2])))
        result[str(doc_id)] = spans
    return result

def exp6_report(name, texts, gold):
    gold_norm = exp6_normalize_gold(gold)
    counts = Counter()
    docs_with = Counter()
    lengths = []
    per_doc = defaultdict(Counter)

    for doc_id, text in texts.items():
        did = str(doc_id)
        lengths.append(len(text))
        for s, e, lab in gold_norm.get(did, []):
            counts[lab] += 1
            per_doc[did][lab] += 1
            if per_doc[did][lab] == 1:
                docs_with[lab] += 1

    print(f"\n=== {name} ===")
    print("Documents:", len(texts))
    print("Mean chars/document:", round(float(np.mean(lengths)), 1))
    print("Median chars/document:", int(np.median(lengths)))
    print(f"{'LABEL':12s} {'SPANS':>8s} {'DOCS':>8s} {'SPANS/100K CHARS':>18s}")
    print("-" * 52)

    total_chars = sum(lengths) or 1
    for lab in ENTITY_LABELS:
        print(
            f"{lab:12s} {counts[lab]:8d} {docs_with[lab]:8d} "
            f"{counts[lab] / total_chars * 100000:18.2f}"
        )

    return gold_norm, counts, per_doc

exp6_train_norm, exp6_train_counts, exp6_train_doc_counts = exp6_report(
    "TRAIN FOLD", train_texts, train_gold
)
exp6_val_norm, exp6_val_counts, exp6_val_doc_counts = exp6_report(
    "VALIDATION FOLD", val_texts, val_gold
)

print("\n=== VALIDATION / TRAIN ENTITY FREQUENCY RATIO ===")
for lab in ENTITY_LABELS:
    tr = exp6_train_counts[lab]
    va = exp6_val_counts[lab]
    print(f"{lab:12s} train={tr:5d} val={va:5d} ratio={va/max(tr,1):.3f}")


In [ ]:

# EXPERIMENT 7 — Boundary and label-confusion audit
# Uses raw model probabilities with the existing thresholds.

exp7_pred = apply_thr(exp1_raw_val, champ_thr)
exp7_labels = ["PREC", "CASENO", "STAT", "AUTH"]

def exp7_audit_label(label, max_examples=10):
    gold_set, pred_set = exp3_span_sets(val_gold, exp7_pred, label)
    exact = gold_set & pred_set
    fp = sorted(pred_set - gold_set)
    fn = sorted(gold_set - pred_set)

    print(f"\n{'='*80}\n{label}: exact={len(exact)}, FP={len(fp)}, FN={len(fn)}")

    print("\nMISSED GOLD SPANS WITH OVERLAPPING PREDICTIONS")
    shown = 0
    for did, gs, ge, glab in fn:
        text = val_texts[did]
        overlaps = [
            (ps, pe, plab)
            for pdid, ps, pe, plab in pred_set
            if pdid == did and ps < ge and pe > gs
        ]
        if overlaps:
            print(f"\nGold [{gs}:{ge}] {text[gs:ge]!r}")
            print("Overlapping predictions:", [
                (ps, pe, plab, text[ps:pe])
                for ps, pe, plab in overlaps
            ])
            shown += 1
            if shown >= max_examples:
                break

    if shown == 0:
        print("No overlapping predictions found in the displayed sample.")

    print("\nFALSE POSITIVE SAMPLE")
    for did, ps, pe, plab in fp[:max_examples]:
        text = val_texts[did]
        print(f"{text[max(0,ps-80):min(len(text),pe+80)].replace(chr(10),' ')}")
        print(f"  Predicted: {text[ps:pe]!r} [{ps}:{pe}]")

for lab in exp7_labels:
    exp7_audit_label(lab)


In [ ]:

# EXPERIMENT 8 — Per-document validation diagnostics
# Robust to extra fields in gold/prediction records.
# Validation only; no training and no files written.

import numpy as np

exp8_pred = apply_thr(exp1_raw_val, champ_thr)

def exp8_normalize_spans(records):
    """Convert span records to (start, end, label), ignoring extra fields."""
    out = set()
    for rec in records:
        if len(rec) < 3:
            continue
        s, e, label = rec[0], rec[1], rec[2]
        out.add((int(s), int(e), str(label)))
    return out

def exp8_get_doc_records(mapping, doc_id):
    # Supports mappings keyed by either string or original document IDs.
    if doc_id in mapping:
        return mapping[doc_id]
    return mapping.get(str(doc_id), [])

def exp8_doc_metrics(doc_id, gold_by_doc, pred_by_doc):
    gold_spans = exp8_normalize_spans(
        exp8_get_doc_records(gold_by_doc, doc_id)
    )
    pred_spans = exp8_normalize_spans(
        exp8_get_doc_records(pred_by_doc, doc_id)
    )

    per_label = {}
    for label in ENTITY_LABELS:
        g = {x for x in gold_spans if x[2] == label}
        p = {x for x in pred_spans if x[2] == label}
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        denom = 2 * tp + fp + fn
        per_label[label] = 2 * tp / denom if denom else 0.0

    return (
        float(np.mean(list(per_label.values()))),
        per_label,
        len(gold_spans),
        len(pred_spans),
    )

exp8_rows = []
for doc_id, text in val_texts.items():
    score, per_label, n_gold, n_pred = exp8_doc_metrics(
        doc_id, val_gold, exp8_pred
    )
    exp8_rows.append({
        "doc_id": str(doc_id),
        "chars": len(text),
        "macro_f1": score,
        "gold_spans": n_gold,
        "pred_spans": n_pred,
        **{lab: per_label.get(lab, 0.0)
           for lab in ["APP", "RESP", "WIT", "AUTH", "STAT", "PREC", "CASENO"]},
    })

exp8_rows.sort(key=lambda row: row["macro_f1"])

print("=== 10 LOWEST-SCORING VALIDATION DOCUMENTS ===")
for row in exp8_rows[:10]:
    print(row)

print("\n=== 10 HIGHEST-SCORING VALIDATION DOCUMENTS ===")
for row in exp8_rows[-10:][::-1]:
    print(row)

scores = [row["macro_f1"] for row in exp8_rows]
print("\n=== SUMMARY ===")
print("Documents:", len(exp8_rows))
print("Mean per-document macro-F1:", round(float(np.mean(scores)), 4))
print("Median per-document macro-F1:", round(float(np.median(scores)), 4))
print("Documents with zero per-document macro-F1:", sum(s == 0 for s in scores))


In [ ]:

# EXPERIMENT 9 — Self-contained span consistency audit
# Validation only. No training and no files written.

from collections import Counter

def exp9_normalize(records):
    out = set()
    for rec in records:
        if len(rec) >= 3:
            out.add((int(rec[0]), int(rec[1]), str(rec[2])))
    return out

def exp9_lookup(mapping, doc_id):
    if doc_id in mapping:
        return mapping[doc_id]
    return mapping.get(str(doc_id), [])

def exp9_overlaps(spans):
    ordered = sorted(spans, key=lambda x: (x[0], x[1]))
    count = 0
    for i, (s, e, lab) in enumerate(ordered):
        if any(
            other_s < e and other_e > s
            for j, (other_s, other_e, _) in enumerate(ordered)
            if i != j
        ):
            count += 1
    return count

# Use the current raw model output and threshold configuration.
exp9_raw = predict_texts(champion_model, val_texts)
exp9_pred = apply_thr(exp9_raw, champ_thr)

exp9_totals = {
    "gold": Counter(),
    "pred": Counter(),
}

for doc_id in val_texts:
    gold_spans = exp9_normalize(exp9_lookup(val_gold, doc_id))
    pred_spans = exp9_normalize(exp9_lookup(exp9_pred, doc_id))

    exp9_totals["gold"]["total"] += len(gold_spans)
    exp9_totals["pred"]["total"] += len(pred_spans)
    exp9_totals["gold"]["overlapping_spans"] += exp9_overlaps(gold_spans)
    exp9_totals["pred"]["overlapping_spans"] += exp9_overlaps(pred_spans)

    for s, e, label in gold_spans:
        exp9_totals["gold"][label] += 1
    for s, e, label in pred_spans:
        exp9_totals["pred"][label] += 1

print("=== SPAN CONSISTENCY ===")
for kind in ("gold", "pred"):
    print(
        f"{kind.upper()}: total={exp9_totals[kind]['total']}, "
        f"spans participating in overlaps="
        f"{exp9_totals[kind]['overlapping_spans']}"
    )

print("\n=== GOLD VS PREDICTED COUNTS ===")
print(f"{'LABEL':12s} {'GOLD':>8s} {'PRED':>8s} {'PRED/GOLD':>11s}")
for label in ENTITY_LABELS:
    g = exp9_totals["gold"][label]
    p = exp9_totals["pred"][label]
    print(f"{label:12s} {g:8d} {p:8d} {p/max(g,1):11.3f}")

print("\n=== EXACT-SPAN METRICS ===")
exp9_metrics = calculate_exact_span_metrics(
    val_gold, exp9_pred, ENTITY_LABELS
)
print("Macro-F1:", round(exp9_metrics["MACRO_F1"], 6))
for label in ENTITY_LABELS:
    m = exp9_metrics[label]
    print(
        f"{label:12s} P={m['precision']:.4f} "
        f"R={m['recall']:.4f} F1={m['f1']:.4f} "
        f"TP={m['TP']} FP={m['FP']} FN={m['FN']}"
    )


In [ ]:

# EXPERIMENT 10 — Diagnose the threshold/pipeline mismatch
# Validation only. No training, no file writes.

VERIFIED_CHAMPION_THR = {
    "APP": 0.15,
    "RESP": 0.45,
    "A.COUNSEL": 0.45,
    "R.COUNSEL": 0.45,
    "JUDGE": 0.25,
    "WIT": 0.45,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.25,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Reuse the same raw predictions for a fair comparison.
exp10_raw = predict_texts(champion_model, val_texts)

def exp10_score(name, pred):
    m = calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)
    print(f"\n{name}: macro-F1={m['MACRO_F1']:.6f}")
    for lab in ENTITY_LABELS:
        x = m[lab]
        print(
            f"{lab:12s} P={x['precision']:.3f} "
            f"R={x['recall']:.3f} F1={x['f1']:.3f} "
            f"TP={x['TP']} FP={x['FP']} FN={x['FN']}"
        )
    return m

print("Active thresholds:", champ_thr)
print("Verified thresholds:", VERIFIED_CHAMPION_THR)

exp10_active_pred = apply_thr(exp10_raw, champ_thr)
exp10_verified_pred = apply_thr(exp10_raw, VERIFIED_CHAMPION_THR)

exp10_active_metrics = exp10_score(
    "ACTIVE THRESHOLDS — RAW MODEL", exp10_active_pred
)
exp10_verified_metrics = exp10_score(
    "VERIFIED THRESHOLDS — RAW MODEL", exp10_verified_pred
)

# Original rule merge, using the same fold-1-selected rule configuration.
exp10_verified_merged = merge_all(
    val_texts, exp10_verified_pred, keep
)
exp10_merged_metrics = exp10_score(
    "VERIFIED THRESHOLDS + ORIGINAL RULES", exp10_verified_merged
)

print("\n=== SUMMARY ===")
print("Active raw:", round(exp10_active_metrics["MACRO_F1"], 6))
print("Verified raw:", round(exp10_verified_metrics["MACRO_F1"], 6))
print("Verified + rules:", round(exp10_merged_metrics["MACRO_F1"], 6))


In [ ]:

# EXPERIMENT 11 — Verified baseline vs complete WIT hybrid
# Validation only. No training or file writes.

# Start from the established champion thresholds and original rules.
exp11_raw = predict_texts(champion_model, val_texts)
exp11_thr = apply_thr(exp11_raw, VERIFIED_CHAMPION_THR)
exp11_base = merge_all(val_texts, exp11_thr, keep)

exp11_base_metrics = calculate_exact_span_metrics(
    val_gold, exp11_base, ENTITY_LABELS
)

# Use the previously validated context-filter variant as the starting point.
if "_wit_filter_results" not in globals() or "name_shape_plus_context" not in _wit_filter_results:
    raise RuntimeError(
        "Missing validated WIT filter results. Run the WIT candidate/filter cells first."
    )

# Reuse the existing validation candidate output.
exp11_wit_predictions, exp11_removed = apply_final_wit_rule(
    _wit_filter_results["name_shape_plus_context"][0],
    val_texts
)
exp11_wit_metrics = calculate_exact_span_metrics(
    val_gold, exp11_wit_predictions, ENTITY_LABELS
)

print("=== VERIFIED BASELINE ===")
print("Macro-F1:", round(exp11_base_metrics["MACRO_F1"], 6))

print("\n=== WIT HYBRID ===")
print("Macro-F1:", round(exp11_wit_metrics["MACRO_F1"], 6))
print("Delta:", round(
    exp11_wit_metrics["MACRO_F1"] - exp11_base_metrics["MACRO_F1"], 6
))
w = exp11_wit_metrics["WIT"]
print(
    f"WIT P={w['precision']:.4f} R={w['recall']:.4f} "
    f"F1={w['f1']:.4f} TP={w['TP']} FP={w['FP']} FN={w['FN']}"
)

print("\n=== PER-LABEL DELTAS ===")
for lab in ENTITY_LABELS:
    b = exp11_base_metrics[lab]["f1"]
    n = exp11_wit_metrics[lab]["f1"]
    print(f"{lab:12s} {b:.4f} -> {n:.4f} ({n-b:+.4f})")


In [ ]:
# EXPERIMENT 12 — Corrected test inference with the complete WIT hybrid
# Does not train or overwrite any existing prediction file.

import json
import re
from pathlib import Path

_REQUIRED = [
    "champion_model", "test_texts", "predict_texts", "apply_thr",
    "merge_all", "keep", "VERIFIED_CHAMPION_THR", "ENTITY_LABELS",
    "candidates_for_text", "_WIT_ID", "_valid_person_surface",
    "_candidate_surface",
]
_missing = [x for x in _REQUIRED if x not in globals()]
if _missing:
    raise RuntimeError("Run the notebook setup, original rule, and WIT candidate cells first. Missing: "
                       + ", ".join(_missing))

FINAL_WIT_EXCLUSIONS = {
    "associate professor",
    "both pw",
    "manchili",
    "oblique",
    "subsequently",
    "the police patil",
}

def test_near_wit_id(text, s, e, radius=60):
    region = text[max(0, s-radius):min(len(text), e+radius)]
    return bool(_WIT_ID.search(region))

# 1. Fresh baseline with the VERIFIED thresholds.
champion_model.eval()
test_raw_exp12 = predict_texts(champion_model, test_texts)
test_thr_exp12 = apply_thr(test_raw_exp12, VERIFIED_CHAMPION_THR)
test_base_exp12 = merge_all(test_texts, test_thr_exp12, keep)

# Normalize document keys and span records.
test_final_exp12 = {
    str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
    for d, spans in test_base_exp12.items()
}
test_texts_exp12 = {str(d): text for d, text in test_texts.items()}

# 2. Generate and filter new WIT candidates.
exp12_counts = {"candidates": 0, "rejected": 0, "blocked": 0,
                "excluded": 0, "added": 0}

for doc_id, text in test_texts_exp12.items():
    pred = test_final_exp12.setdefault(doc_id, set())

    for s, e, label in candidates_for_text(text):
        if str(label) != "WIT":
            continue

        s, e = int(s), int(e)
        exp12_counts["candidates"] += 1
        surface = _candidate_surface(text, s, e)

        # Match the validated name_shape_plus_context logic.
        if not test_near_wit_id(text, s, e, radius=60):
            exp12_counts["rejected"] += 1
            continue

        if not _valid_person_surface(surface, "name_shape_plus_context"):
            exp12_counts["rejected"] += 1
            continue

        if not test_near_wit_id(text, s, e, radius=30):
            exp12_counts["rejected"] += 1
            continue

        if surface.casefold() in FINAL_WIT_EXCLUSIONS:
            exp12_counts["excluded"] += 1
            continue

        # Do not add a candidate that overlaps any existing entity.
        if any(max(s, ps) < min(e, pe) for ps, pe, _ in pred):
            exp12_counts["blocked"] += 1
            continue

        pred.add((s, e, "WIT"))
        exp12_counts["added"] += 1

# 3. Validate document IDs, labels, and character offsets.
test_final_exp12 = {
    d: sorted(spans) for d, spans in test_final_exp12.items()
}
assert set(test_final_exp12) == set(test_texts_exp12)

for doc_id, spans in test_final_exp12.items():
    text = test_texts_exp12[doc_id]
    for s, e, label in spans:
        assert 0 <= s < e <= len(text), (doc_id, s, e, len(text))
        assert label in ENTITY_LABELS, (doc_id, label)

# 4. Save separately. Existing champion predictions are untouched.
out_path = Path("/kaggle/working/prediction_wit_hybrid_verified.json")
serialized = {
    doc_id: [[int(s), int(e), str(label)] for s, e, label in spans]
    for doc_id, spans in test_final_exp12.items()
}

with out_path.open("w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False)

print("Saved:", out_path)
print("Documents:", len(serialized))
print("Total predicted spans:", sum(map(len, serialized.values())))
print("WIT candidate audit:", exp12_counts)
print("Existing champion prediction file was not overwritten.")
print("No test gold labels were used.")

In [ ]:
# EXPERIMENT 13 — Conservative entity-boundary correction
# Validation only. Tests span trimming; no model training or file writes.

import re

_REQUIRED = [
    "champion_model", "val_texts", "val_gold", "predict_texts",
    "apply_thr", "merge_all", "keep", "VERIFIED_CHAMPION_THR",
    "val_rule_candidates", "calculate_exact_span_metrics",
    "ENTITY_LABELS", "_WIT_ID", "_valid_person_surface",
    "_candidate_surface",
]
_missing = [x for x in _REQUIRED if x not in globals()]
if _missing:
    raise RuntimeError("Run setup and WIT cells first. Missing: "
                       + ", ".join(_missing))

FINAL_WIT_EXCLUSIONS = {
    "associate professor", "both pw", "manchili",
    "oblique", "subsequently", "the police patil",
}

# Rebuild a fresh verified baseline.
exp13_raw = predict_texts(champion_model, val_texts)
exp13_thr = apply_thr(exp13_raw, VERIFIED_CHAMPION_THR)
exp13_base = merge_all(val_texts, exp13_thr, keep)

# Add WIT candidates using the validated filter logic.
exp13_hybrid = {
    str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
    for d, spans in exp13_base.items()
}

def exp13_near_wit_id(text, s, e, radius=60):
    return bool(_WIT_ID.search(
        text[max(0, s-radius):min(len(text), e+radius)]
    ))

for d, candidates in val_rule_candidates.items():
    d = str(d)
    text = val_texts[d]
    pred = exp13_hybrid.setdefault(d, set())

    for s, e, label in candidates:
        if str(label) != "WIT":
            continue
        s, e = int(s), int(e)
        surface = _candidate_surface(text, s, e)

        if not exp13_near_wit_id(text, s, e, 60):
            continue
        if not _valid_person_surface(surface, "name_shape_plus_context"):
            continue
        if not exp13_near_wit_id(text, s, e, 30):
            continue
        if surface.casefold() in FINAL_WIT_EXCLUSIONS:
            continue
        if any(max(s, ps) < min(e, pe) for ps, pe, _ in pred):
            continue

        pred.add((s, e, "WIT"))

exp13_hybrid = {d: sorted(spans) for d, spans in exp13_hybrid.items()}
exp13_base_metrics = calculate_exact_span_metrics(
    val_gold, exp13_hybrid, ENTITY_LABELS
)

# Only these labels are tested for trimming.
EXP13_TARGET_LABELS = ["PREC", "CASENO", "AUTH", "STAT"]

def exp13_trim_span(text, s, e):
    # Trim whitespace and punctuation only when they are inside the span.
    trim_chars = " \t\r\n,.;:()[]{}"
    while s < e and text[s] in trim_chars:
        s += 1
    while e > s and text[e-1] in trim_chars:
        e -= 1
    return (s, e)

def exp13_apply_trim(predictions, labels):
    output = {}
    changed = 0

    for d, spans in predictions.items():
        text = val_texts[d]
        new_spans = set()

        for s, e, lab in spans:
            if lab in labels:
                ns, ne = exp13_trim_span(text, s, e)
                if (ns, ne) != (s, e):
                    changed += 1
                if ns < ne:
                    new_spans.add((ns, ne, lab))
            else:
                new_spans.add((s, e, lab))

        output[d] = sorted(new_spans)

    return output, changed

print("=== BASELINE: VERIFIED RULES + WIT HYBRID ===")
print("Macro-F1:", round(exp13_base_metrics["MACRO_F1"], 6))

exp13_results = {}
for lab in EXP13_TARGET_LABELS:
    pred, changed = exp13_apply_trim(exp13_hybrid, {lab})
    metrics = calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)
    exp13_results[lab] = (metrics["MACRO_F1"], changed, metrics)

    print(
        f"\nTrim {lab}: macro={metrics['MACRO_F1']:.6f}, "
        f"delta={metrics['MACRO_F1'] - exp13_base_metrics['MACRO_F1']:+.6f}, "
        f"spans changed={changed}, label F1={metrics[lab]['f1']:.4f}"
    )

exp13_all_pred, exp13_all_changed = exp13_apply_trim(
    exp13_hybrid, set(EXP13_TARGET_LABELS)
)
exp13_all_metrics = calculate_exact_span_metrics(
    val_gold, exp13_all_pred, ENTITY_LABELS
)

print("\n=== COMBINED TRIM ===")
print("Macro-F1:", round(exp13_all_metrics["MACRO_F1"], 6))
print("Delta:", round(
    exp13_all_metrics["MACRO_F1"] - exp13_base_metrics["MACRO_F1"], 6
))
print("Spans changed:", exp13_all_changed)

print("\n=== PER-LABEL F1: BASELINE VS COMBINED ===")
for lab in ENTITY_LABELS:
    old = exp13_base_metrics[lab]["f1"]
    new = exp13_all_metrics[lab]["f1"]
    print(f"{lab:12s} {old:.4f} -> {new:.4f} ({new-old:+.4f})")

print("\nNo files written. No model weights changed.")

In [ ]:
# EXPERIMENT 14 — CASENO boundary ablation
# Validation only. Does not modify the model or write files.

import string

assert "exp13_hybrid" in globals(), "Run Experiment 13 first."
assert "exp13_base_metrics" in globals(), "Run Experiment 13 first."

BASELINE_F1 = exp13_base_metrics["MACRO_F1"]

WS = " \t\r\n"
PUNCT = ",.;:()[]{}"
ALL_TRIM = WS + PUNCT

def exp14_transform_span(text, s, e, mode):
    left, right = s, e

    if mode == "both":
        left_chars = right_chars = ALL_TRIM
    elif mode == "leading":
        left_chars, right_chars = ALL_TRIM, ""
    elif mode == "trailing":
        left_chars, right_chars = "", ALL_TRIM
    elif mode == "whitespace":
        left_chars = right_chars = WS
    elif mode == "punctuation":
        left_chars = right_chars = PUNCT
    elif mode == "trailing_punctuation":
        left_chars, right_chars = "", PUNCT
    elif mode == "leading_punctuation":
        left_chars, right_chars = PUNCT, ""
    else:
        raise ValueError(mode)

    while left < right and text[left] in left_chars:
        left += 1
    while right > left and text[right - 1] in right_chars:
        right -= 1

    return left, right

def exp14_make_predictions(labels_to_trim, mode):
    out = {}
    changed = 0

    for doc_id, spans in exp13_hybrid.items():
        text = val_texts[doc_id]
        new_spans = set()

        for s, e, lab in spans:
            if lab in labels_to_trim:
                ns, ne = exp14_transform_span(text, s, e, mode)
                changed += (ns, ne) != (s, e)
                if ns < ne:
                    new_spans.add((ns, ne, lab))
            else:
                new_spans.add((s, e, lab))

        out[doc_id] = sorted(new_spans)

    return out, changed

exp14_tests = [
    ("CASENO: both", {"CASENO"}, "both"),
    ("CASENO: leading", {"CASENO"}, "leading"),
    ("CASENO: trailing", {"CASENO"}, "trailing"),
    ("CASENO: whitespace only", {"CASENO"}, "whitespace"),
    ("CASENO: punctuation only", {"CASENO"}, "punctuation"),
    ("CASENO: leading punctuation", {"CASENO"}, "leading_punctuation"),
    ("CASENO: trailing punctuation", {"CASENO"}, "trailing_punctuation"),
    ("CASENO + STAT: both", {"CASENO", "STAT"}, "both"),
]

exp14_results = []

for name, labs, mode in exp14_tests:
    pred, changed = exp14_make_predictions(labs, mode)
    metrics = calculate_exact_span_metrics(val_gold, pred, ENTITY_LABELS)
    score = metrics["MACRO_F1"]

    exp14_results.append({
        "name": name,
        "score": score,
        "delta": score - BASELINE_F1,
        "changed": changed,
        "metrics": metrics,
        "predictions": pred,
    })

exp14_results.sort(key=lambda x: x["score"], reverse=True)

print(f"Reference WIT hybrid: {BASELINE_F1:.6f}")
print("\n=== CASENO BOUNDARY ABLATION ===")
for r in exp14_results:
    print(
        f"{r['name']:30s} "
        f"macro={r['score']:.6f} "
        f"delta={r['delta']:+.6f} "
        f"changed={r['changed']}"
    )
    print(
        f"  CASENO F1={r['metrics']['CASENO']['f1']:.4f}, "
        f"STAT F1={r['metrics']['STAT']['f1']:.4f}"
    )

best = exp14_results[0]
print("\nBest variant:", best["name"])
print(f"Best macro-F1: {best['score']:.6f}")
print(f"Gain over WIT hybrid: {best['delta']:+.6f}")

if best["score"] > 0.333194:
    print("This beats the previous CASENO-only result.")
elif best["score"] > BASELINE_F1:
    print("This improves on the WIT hybrid, but does not beat CASENO-only trimming.")
else:
    print("No tested variant beats the WIT hybrid baseline.")

print("\nNo files written; champion checkpoint and predictions remain unchanged.")

In [ ]:
# EXPERIMENT 15 — Single-cell recovery + safe fine-tuning preflight
# Does not overwrite the champion checkpoint or existing prediction files.

import os, json, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2Model
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

# Find checkpoint, tokenizer and fold data.
all_files = [p for root in ROOTS if root.exists()
             for p in root.rglob("*") if p.is_file()]

def choose_file(name, preferred=()):
    matches = [p for p in all_files if p.name == name]
    if not matches:
        raise FileNotFoundError(
            f"Missing {name}. Ensure the CS781 recovery dataset is attached."
        )
    preferred_matches = [
        p for p in matches
        if all(token.lower() in str(p).lower() for token in preferred)
    ]
    return sorted(
        preferred_matches or matches,
        key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p))
    )[0]

def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

champion_candidates = [
    p for p in all_files
    if p.name in {"champion_recovery.pt", "best_ner_model.pt"}
]
if not champion_candidates:
    raise FileNotFoundError("Champion checkpoint not found in Kaggle input/working.")

CHAMPION_PATH = sorted(
    champion_candidates,
    key=lambda p: (
        0 if p.name == "champion_recovery.pt" else 1,
        0 if "cs781_recovery" in str(p).lower() else 1,
        str(p)
    )
)[0]

phase1_candidates = [
    p for p in all_files
    if p.name == "checkpoint_latest.pt"
    and ("cs781-phase-1-checkpoints" in str(p)
         or "phase-1" in str(p).lower())
]
if not phase1_candidates:
    raise FileNotFoundError(
        "Phase-1 encoder checkpoint_latest.pt not found. "
        "Attach the Phase-1 checkpoint dataset."
    )

PHASE1_PATH = sorted(phase1_candidates, key=str)[0]
recovery = load_pt(CHAMPION_PATH)
phase1_ckpt = load_pt(PHASE1_PATH)

BIO_LABELS = list(recovery["bio_labels"])
label2id = {x: i for i, x in enumerate(BIO_LABELS)}
id2label = {i: x for i, x in enumerate(BIO_LABELS)}
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
assert len(BIO_LABELS) == 25

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise RuntimeError("Phase-1 checkpoint has no model_config; stopping safely.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.hidden_size, num_labels
        )

    def forward(self, input_ids, attention_mask=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return self.classifier(self.dropout(out.last_hidden_state))

phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)

lm_wrapper = type(
    "LMWrapper", (), {"transformer": phase1_lm, "config": gpt2_config}
)()
champion_model = LegalNERModel(lm_wrapper, len(BIO_LABELS)).to(device)
champion_model.load_state_dict(recovery["model_state_dict"], strict=True)
champion_model.eval()

tokenizer = Tokenizer.from_file(
    str(choose_file("legal_bpe_32k.json"))
)

def load_json_file(name):
    with open(choose_file(name), "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = {str(k): v for k, v in load_json_file("fold1-text.json").items()}
train_gold = {str(k): v for k, v in load_json_file("fold1-gold.json").items()}
val_texts = {str(k): v for k, v in load_json_file("fold2-text.json").items()}
val_gold = {str(k): v for k, v in load_json_file("fold2-gold.json").items()}
test_texts = {str(k): v for k, v in load_json_file("fold3-text.json").items()}

# The verified champion thresholds.
VERIFIED_CHAMPION_THR = {
    "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45,
    "R.COUNSEL": 0.45, "JUDGE": 0.25, "WIT": 0.45,
    "AUTH": 0.35, "COURT": 0.45, "STAT": 0.25,
    "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35
}

# Inference and exact-span metrics.
from bisect import bisect_left, bisect_right

NUM_LABELS = len(BIO_LABELS)
legal_start = np.array(
    [x == "O" or x.startswith("B-") for x in BIO_LABELS], dtype=bool
)
legal_transition = np.ones((NUM_LABELS, NUM_LABELS), dtype=bool)
for pi, prev in enumerate(BIO_LABELS):
    for ci, curr in enumerate(BIO_LABELS):
        if curr.startswith("I-"):
            legal_transition[pi, ci] = prev in (
                f"B-{curr[2:]}", f"I-{curr[2:]}"
            )

def bio_viterbi_decode(logits):
    emissions = np.asarray(logits, dtype=np.float64)
    if not len(emissions):
        return []
    neg_inf = -1e12
    scores = np.full_like(emissions, neg_inf)
    back = np.zeros(emissions.shape, dtype=np.int32)
    scores[0, legal_start] = emissions[0, legal_start]
    for t in range(1, len(emissions)):
        cand = np.where(
            legal_transition, scores[t-1][:, None], neg_inf
        ) + emissions[t][None, :]
        back[t] = np.argmax(cand, axis=0)
        scores[t] = cand[back[t], np.arange(NUM_LABELS)]
    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return list(reversed(path))

def decode_spans(text, offsets, pred_ids):
    spans, current_label, current_start, current_end = [], None, None, None

    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            s, e = current_start, current_end
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e-1].isspace():
                e -= 1
            if s < e:
                spans.append((s, e, current_label))
        current_label = current_start = current_end = None

    for (s, e), pred_id in zip(offsets, pred_ids):
        if e <= s:
            continue
        tag = BIO_LABELS[int(pred_id)]
        if tag == "O":
            close()
            continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or current_label != lab:
            close()
            current_label, current_start, current_end = lab, s, e
        else:
            current_end = e
    close()
    return sorted(set(spans))

@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    if not ids:
        return offsets, torch.zeros((0, len(BIO_LABELS)))
    sums = torch.zeros((len(ids), len(BIO_LABELS)))
    counts = torch.zeros(len(ids))
    model.eval()
    for start in range(0, len(ids), 448):
        end = min(start + 512, len(ids))
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        logits = model(x, torch.ones_like(x))[0].float().cpu()
        sums[start:end] += logits
        counts[start:end] += 1
        if end == len(ids):
            break
    return offsets, torch.softmax(sums / counts.clamp_min(1).unsqueeze(-1), -1)

def predict_texts(model, texts):
    result = {}
    model.eval()
    for did, text in tqdm(texts.items(), desc="Validation inference"):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets):
            result[str(did)] = []
            continue
        ids = bio_viterbi_decode(torch.log(probs.clamp_min(1e-12)).numpy())
        spans = []
        starts = [a for a, b in offsets]
        ends = [b for a, b in offsets]
        for s, e, lab in decode_spans(text, offsets, ids):
            ix = [
                i for i in range(bisect_right(ends, s), bisect_left(starts, e))
                if ends[i] > starts[i]
            ]
            conf = float(probs[ix, ids[ix]].mean()) if ix else 0.0
            spans.append((s, e, lab, conf))
        result[str(did)] = spans
    return result

def apply_thr(raw, thresholds):
    return {
        d: [(s, e, lab) for s, e, lab, conf in spans
            if conf >= thresholds.get(lab, 0.0)]
        for d, spans in raw.items()
    }

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels=ENTITY_LABELS):
    metrics = {}
    for lab in labels:
        gs, ps = set(), set()
        for d, ents in gold_by_doc.items():
            for ent in ents:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    gs.add((str(d), int(ent[0]), int(ent[1]), lab))
        for d, spans in pred_by_doc.items():
            for ent in spans:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    ps.add((str(d), int(ent[0]), int(ent[1]), lab))
        tp, fp, fn = len(gs & ps), len(ps - gs), len(gs - ps)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        metrics[lab] = {
            "gold": len(gs), "pred": len(ps), "TP": tp, "FP": fp, "FN": fn,
            "precision": p, "recall": r, "f1": f
        }
    metrics["MACRO_F1"] = sum(metrics[x]["f1"] for x in labels) / len(labels)
    return metrics

print("Recovered model, tokenizer and data.")
print("Documents:", len(train_texts), len(val_texts), len(test_texts))
print("Champion:", CHAMPION_PATH)

# We cannot safely recreate the fold-1-selected counsel/judge rules from
# checkpoint/data files alone: they were defined in other notebook cells.
# Stop before training if their saved source/config is not present.
required_rule_objects = [
    "merge_all", "keep", "val_rule_candidates",
    "_WIT_ID", "_valid_person_surface", "_candidate_surface"
]
missing_rules = [x for x in required_rule_objects if x not in globals()]

if missing_rules:
    raise RuntimeError(
        "Setup recovery succeeded, but the exact validated hybrid rule "
        "definitions are not available in this cell: "
        + ", ".join(missing_rules)
        + ". I am stopping before training so we do not compare a fine-tune "
          "against the wrong evaluator. Those rule definitions must be included "
          "to make this experiment valid."
    )

print("Rule prerequisites found. Safe to proceed with the fine-tuning cell.")

In [ ]:

# EXPERIMENT 15 — All-in-one recovery + controlled class-weighted fine-tune
# Runs the exact setup/rule definitions extracted from this notebook, then trains
# a separate copy of champion_model. Existing champion/prediction files untouched.

import base64, zlib, json, copy, gc
from pathlib import Path

_SETUP_PAYLOAD = "eNrNfQtb20iW6F+ppeebSMFWsPOYHqc9WQKkQzoBPkM6M2t7/cm2DGpsySPJPBr47/c86inJhmTvfvf27gRLqjr1OnXedap/tzWJ5vOtjmg1xFaerrJJBA9bP4lWII6iSZTnYXYr4sUyzYpcpMn8VmyLbJUU8SIaJPxepHlD/JGnSUNkUUOcTxpiERYXDTFJl7eDZJalC7GEF/N4LCGJE3iUXybpfB5NijhNcvV1L4UGoqwhptEsXM2LaTwpBoluLlktlrcizEWy1O+KNJtcuE9BklChRDZUpJdREv8ZZbqdM/VGlcjCJJ+l2cIq8+vJWXsvTWbxeYN+f0mn0VyV//d0EYSrIlWF8QV2dBpdxZNIdGVH+NEbbE1W03CwJeKZ/IDPQZyPwqswnofjeeT5IprnkYCiy9Vgyx8kyyxOCqi6TzA6gy2cFfwJ3wAO/w6K2yU01xWqic4gEfCfqvzryVeqabV6HhUjrjxKwkXk7fj+1kNDaHxol/ChHYjD5CpKAATM/XwudJ/FNCzCPCryF5OLaHK5TKHNXIwjmMhITC7SNI+TcxEmt8UF/BgkvePjs1OYmz4iAXTuxWV4fj6PXsTJclXAmBui9OE6zS6x5pY/HCR7Hw/2fjs5Pjw6Gx3tfjlAQHc8WBj7RbhYAiKNsmiSXkXZbbAscNiDrXGUF6MkykYLXD792nR4NA8LKMNfFLwizGCaouloGc/TYvTzKC+iZe4CdQGap0HyMEj2d892Tw9qejqPzsP5aLyMRi/blwHuHa4/S+fTVrOIborKy3P4V71UYPBLu6542ykuX750SupBZmGcWCVpGviRxgBrPZrF8yjHJYMFgGUVWZoWIk4ELaXENUBGfB1EN3Fe5J4vX+N/GgR8LKJk6i0FQlkiCKqTnc/TMaz4c1hjBLTETYE1PN/H/WT3IYeNFk09QDhPv8ZS1lrq7lrtGBAEH3EeX1fQCcsH+Wo2i2+CeXodZbAlodzdYEsvMvy4GGw9wFxIzP+eBl2UgMIeLw6ShUSW0w2HybTaG9rmcoVwRxga0YVPZkDi8Oj3g6Oz496/oEqXiAn0pDxNcpl0t9d8N9RkNti6Wz4I0b+DnhVh4fnwZ5QDGRUvRGun/er583YnaM0exJf4/ZCaRZJWokhHqdWS4HmaAdmfBlTDjGkwSHBYctpenB3/dnB0+F8HvZrR6TE4y+I0vKyD/X5377evJ6oJsX/YO9gDwIe4ZcugJfbd5UXm3QTLMAOK6NPs3VRafvBr2v5JnMSTS1FcREJRKaHolohulvN4Ehfz27dimibPChFOJvEU2gBsumViGgFWiDAbx7BxqW6VhAW4GyQpnAAOxVP8UsLPyn4xaMqMpJaYDgmJEtj+NS3I4f7P264h109pGCgZTM8HgHiUFh8Qmw6yLM08g8IS8aojw51YaZUxUsD/okw4bAqLl9hTIJBgmoZ2iyKcXNBC753+7eeWXm6LTSp8wTn5jaAFCojPcgQQ1TC5HOkuoxwFmCQHTPQQMHHpK+ogJyIqVlkirIHv4BTDoubQEz1upjk5ixytRqW0HFlzHE4uV8vHSjsT9GJt6dwa397H3S8nh8dHo5Pds4+GttescENcRrfd6lz4/R0mgRcwj60Sxsndtx7vNOdyEb9OKCBaLCewSa01W01L4JHjpcUYJAZbKx1TNGF9h5/U6e/tuOyy3U0XZ36wy99FUer6R8v3SMtrNzZu5xMeWS0ldHnKyUcg8K0SslXaZFSbh4vxNBTLjvA24TdNpEJy+ehrpDR85vTgMzCVg31xBKxLoT3J5M4esAR+XYN6DeM7ONo73j/oUSVrJHaVSGpSYpkCH7nt1LAYXC3ow9uaaaNvUTIB6pfhlDn6wMuSPvAS9YF8Ce3ZEBZRESJNe8uqIhK/yyS9Tkz7cS5y6mY0VX2BiZxFwEgnEVO8PJxFI1JVRvM0hBXCfS4xochuLYlE0jlWa3RZVD+XUHUS4lx0pS7VENdRfH5R5CPsWvdDCGsmUT+6mUSAK2egRBFe/WALhGN6xruVcVRWWmLeJbRdLe0usVnkPTWT1rQDwuaEF3OQvT3N1vA1SNCIvjEoYSCtwRzrz6BIgl4tkRcVSP3Ft3CqBk3c1qxRrGnQKlFt0/ros4S6sWZJjFS9Q8XsRbRMgeEqFKQe3pmVvOwIu6ug2HmXLLld4k4GYAiEJXxUR0A/My8ItKX18RffIcUExmqCPz5Iqa8XsakEtkRmTWb+THdYXMfFRQqyRQ50LgEZEJBvOQ8nqD3jRlJbKKgIsMdnH4mqaMn/y8HZLoqzVfl1jXiPlJpmwcFRl85Yu2KSJkWcrKK6PZmO/6jB5qU7V9YaQ3l3bfWCATIBJEIyKKSQyy2Ek4eKNawulvnhVU2XRbxAUxDr+M7a2v+pdcZRImN1ByLbbwgPJrghZjD0griC7z+48FBzQgORqiRNByN6PyKDV7kL+UW4jEZAK1Ood5QmkfvZnVWCUz+vtIBzFh2hkGx+Mg/zPJ7FQPqZTtZOASqQUBWIOLJr7ATNAiB9WADvg284mdRTqN+pn0NnIMVqCWo+VAzodalNW+XsEEZ07/DffqfVHgJ7Unune4e/4EVlGN0709yDMyZLK61vDtYG+FT3jugUPPjBiGxlo5ELCNYNyzq85ID+IJkOQQZeo0R3YCOt5lOay1gyU4+bi6zGOuIuAjJSYsivSgz5VQA0BjYmYNsK4BA60eLg/jPGzxdEaqQg+f7wePR59/3B51O10RQL6AOhi1MQpMYwTSCkwXDpdzue4naj3x0R006LgRPgM+6LKFktogzQyjOwEffjaZvLQOW4I8s/tTLo+Ydn/zI97Svb1e7JCW/f3sGp/LUb7B1/PQLhSX5wHz993f/1gH9+OzyzrGC7X88+8nso3zvjn6dnu/LXSe9gj38BWZUA9oA0Hh2jvgbCHmBdBOQdCLfdd5R726+BEAy2Dm6WLPS0XwsowYOG3XIOa39XqvaAQEGHn53DYMtMS1GKCRmltVkHysKuxB3pyMy/RbdKVK7h5BdoH0+FDfCtAEkYETKzkEkKhUTV57dKnD5fFm1ZC/ppDOVS7bwC8WhMhqEu9A4QyrwAhJIzD1pcmsdk/ZelkLdENyDDR8k52tlMyUlx82iZCKR2Wegink6jpNLePLyNMlkE8I2fc7vERRROrQL4aH3PojyejpbTLF12d4KdhsAm7WeghIl5HiQ0V0SXxGc0+4LES04EL0kC+LGaR4pQovA7AkoVF6ORB1LyrAHkIiIDbTQdzRcNwR1GzLGJa75aoh4X6LoWcUIwgVq/rgsvsBwepRrYfZRFugJ6uc8P3k7QKkM29JaLfgbQIShBTjOMI4G1IM5IzOChK9dhNpVjJ0VrFE9zmlMQiVDpX4T5ZRfR3J4A6BwUJZZmDdfTALrrQbmPfkXyL43Ss2fHk+0G8LkYqeEhU2Wztdy684XcHrzs1rYxSoCSB6TAehlk0QIIMczjLL7xyEqvVgq3X0dcGbvtZUNclWTPfp08MQziIlrkCjtQlEELblbkKHlWGyHzvx4DaT8WPM/uOQk58LJ7lq2Qh88Xo+ssXC4JLYihDbY+f/nGr5B8ej4K53aLgy0toCOm0yYnAtcR1ow9+J5vWTeZy3VLG8u03ihTZD8oUk870Fw4lRFazLBmNstjLgGLrsK5p614sziZEj+fsl8D/8FH2uGg/WbwgXh03gUZV6L2IiyAWD/FsQB8RkEc6tVFKi5B2BrtGlsK8Khe9O9VnKFiDl/FnYL4QJDYFqoNlbw7tfFTC0R6OKVeq7FAv2AAXmE7V1xzFNUhk2h5anw5NscCZRqEarIVxwqqjIr/CwYeLVWRLVJ0K8tc7+kjhWw+bY2Iia2tWnEG6nro3HusnuUAlPXaT2ivXdNe+wnttWvae/mE9l6W2rPmFOpoB32A/nauh0vgzjsSW9xjtH0RkmMxQtom0mWUSMvNYIsIEPEIwFzQj1fFrPkzeh5BGppVbT8IkU0/M+4gMjYaGqmnutHSkvqqJE5MTUE9p1AQSMUagG0HIJarBdd2wKHxcw28lw48NEwcAfUN5+i+m6YTkL9BLDzcz0FhIPKWnOeo7eegG5BNIiiPnzxhl8iQHF5kFVJc56E0IRtrEjaZivYEramni5SqbW5NlbA7ac/euk7qMlbFGhsdTn403Wjn1TguIzRszEZsvWdBWqDY5BahGA4jVHu22W5frmXe4Qklcx3sAWtdfAkdFn9KdkxVRs+kLIGD1fX1wO3WzhDqC6hHqDON8z9QvaDeop/ebjSIc/XdasiC9bvuj8BvL2j9ECzRdg3T1EXmp95g4UrHNsKx+6YgGRREWFK1e0Jhp+yGDjp6/OuSHv86AOYMnJZ8/2GWG8OfKC5A+bgACMDg9O/HfF5aXLCcXRsiCqS7SIOvi0sgjVdPsSppe/E7rtWxrrMlS67ABVlqeQkmLgsngGp6yGSB6TjCzWNWREnAja2MtnB0Ky2DNK/Yggx8MbNLz6SdjMpvzVDGt6zAuIauinGTbYLRra9s3zjB8L4P74b0gPKQVefKMR/6LA9dSaMj1QoAk1ZRbgTF0oA10SII3pXv0C4NxVAu2XFl1PyeDtX25an9KHXBWS8pUOMCaHunmXykjqNVTmImfyuHaNThXEXy43aVc1wJ8GbNS67yUniO49ve5GF35VunmvIY+/UeLramW5KNpaGyRN5dt1ccnOSyanEBb7Ff/BInTr5wDG1l5NLL0RD160DgYBO71cZZFF7W20ZtsyhQTukINsvuGrR+Emcg2kcizCLyhyhxBdrWTjczBYIjNKEcBaShEwVIRjRO08vAClOR6HXnRG6gVbEjdoLWa2NbxOdXr0sWRvOyV3lpQ5RmR/zSfq2NjxZIsj7i88vXlg2yBhKbJA0gNkyammyexOfXr20jJZeQLimX8n7b7R0dHv3aQUvgKqeITj2N4tPp8REv7Fv4iLO4edqDdQFfn0kSKi8QCzRVbPJr+Av7HPWquVz0TYmLvgnEHsafatdyg6yu04il/gZthegGtk0zX4aJQG19xVLHRTRfokWQY3zHoC1PChXgy0+jeTQDcigfMvKWAO4eff1iWdNdu8MgYUWQ7C1oKlsGYZaFt16/CM+ZAcMqoQoLz65V5n0TdZMZf8JdasAOgaegdaULaD3XbZBRhWyq3BBsodzzTO8awvz2SxCIimbR1SieNviHbLTWNm8FD05WWUaV6McTKil3kizujvmwWXUflQfX1/2UbQ/ZGqC7jCaN98071UK/3Rk+IArNEH75tZY80O1xBXwpG8cjQpfIm6fncaEtrtEiznMKV0fCjF+CaYSxXp4fMKPzg8lyBf9SoLpXdodzHRWMfRYlwLAkU0CsyBkvVCleHfhAoN+8MsY73Y8A2NtC/EdXtBF/zGtyfPVbQ/xkVrxiCvodGbi2AWk/RT+P/r3CnSOt7LDZDJCHoWlH+jBK7RoXGXQVN4P+TvLmTlVa6A+r82uAhjmZEc1E8PECJ44jic7Rzwf1mq2o1ZYMdgJ0IOedMFuBMDyPLyPP6r2stXaqMfqNq4POleZeaaBWPSBYL9u+3Wx/pyGsjT+0h1T+ZvYSGb8Az88jr9UoTZ7jhg+J/0PL1xdA5bzyDmmoXhTN1rDfaRA3HeoR+2Lb6k0x7OPnhugMTQs49j51mwjW+SK8IUMa0E/gqt0d2x0gm8LCWKQv6za4Ko3GojuyEbScQwUk9aYFCQo6rdz90obUx+L1E+XOUhPmDaa32fIdGaO4CNAsnExRnvW4gw16j43p1iRCSococLg8klFImkowZRgh78g9VPMaIp3N8BQEkc0p+hp0lCYWYioVUXAaKA3mkRbfPEa0pP0hL5b9r3GTTOZpHjkR9kmaYDTS/OmNVIiwrmV790skGEaBToqnQCXT2wXakXPxi+AwAZynfg6KRw5TMsERwMftrmjVVYvEP+CrrhbBAtkVI9GsVISRUGMdnnK11h52uzT/dhyJO/rS8KxnXhpXmyTYvl5zxMc/46W3FhcU7Ra/dEVeFrCdqBrSBVA2sPh9n+LIGaQ/dCDaYkQZLqPL25oG2L9ELnh01CAbXs5jFLuayClbrhLBpbmZ9yytuHMHjAZ+rG3+OzYAlGkQum2M13DryLKyvTofAFphCDV4I+8f7v46+rL7z9Hng6OGoKfTs97h/gEAe91qN8SrVz8Pkv9kRq0lSfLveNLmPI3D8xGIxKNllo5zj1w/DcJYLS4kEzoIpox27JyUNCOcTkcY9xEj3Wb3j83QyGEpcQkHmEwCeoU/5GvJ+aTEiZjmmEiSKp/VuMnjYqbm7VRdZRISSSKjfLXQ59lklaRaRTFDLkd8VHPECZ7qy0tAkk01XEea2nK8LTXhh44nzuLZW40RYxEnHtfaFu6aJxaC3+iuFSSTeX2YzT7V6wCc4dDt6jxFNYJdiV3tUdTCKolv2DZ5JG/UbKMszhLIDXmRXImxVB8n3eoAkkqGa0fi4aSWCrVcWoNzAAy8Y6vi4dX5SPfRrPALCRA93qBm4cS1/GCV5CAKRn9GXrPl7qsSLuXprEAGbqCj0WvRpWq8YZB6xRNpQ7b3iyaRwPpXc6S7dw+b0AA3HQr/WJdsjP+eLjzHJI+rk0+6oCRzm3SGEHE2vIrkNrMtLYZkw06G5jdsbWd+cZvhPpAAqjY5HE6fuzuUh+g2k33FNMoIWaOaKFw896iX9roBx2z75S2JWGsLbaR0IUGh1vohTW0Iui1Z6HhEgPj9ce0HCxBS1dLgaLMS65XmpacITUGRktxVsW0i4xF9GXFmtr+thHs4DGAbfsNW1D09SI5MxkL9eIjiBX2A39UVQSWCtybNax+ZpOoiPAyHwSIKExXrfMPqG4X7OPKSI4Yg88a+4V+UGYjHYVu+E3ZSQheC4Ww6LqI2FECf36JBxMvCa8uYYu0nNsZOO6Lv6fXw3eXhjpCxklYShUJ88Y+ubdxBUzYVhpH6Q96GDVUDV+TacYeRrBrOJys8jzEiawst/mgRYeRE7p1LOzqMVk6veuDQoG6tQVLWtiiEZcLkihbuUBuAY7IBPG3F7hifHUC+i7BT3AwFDcfqnRpWCSf5wETBpo4i79QGzJJmgqcEYS5fkkCLNl94028PSReuikzlvgcgJ3jk+Z4q/IHqO0ProTVkjPKr49HrY83w4wOiWv/XR6TW4AdHVCyloKNmRvxVgwQJc6Y+66Vu6jnEz0m5dtPUts9edbGhF8KDf7cBKG1y+btmp2el8olVPqkpj7Sl/Xz5PHvhLbczPvC8ndUUlIgOWu94SEbqrXPyyHXsMZCTFkch3+sB4fsztFwXS/z5AX/O+OcR/kwa9UvEhmYAM4lz8gp3qBI8h/M5PGV0nryFIB6cDdkfbH3Z3esdjz7ARyJdq4Wnvt0M+1xraE7Lyvg/mDnsuBMNKImWrM2RCr9HWTy7ZautsT3LUz3s21ARhORPCAaJCqvbHMk1SNhPj7qeG8RlBYO53l2O8lOeXa4OJUD4gdIzXGGUg+SH/uXQmAhJ0APEIwjwxQ9CkC78AMv7tC096wADt6ObNvC7jCoU3VKvoMgd+GdlVJ4jxbca4mf/cdHWFmCfXMl0nMP8/5RB/th9glHVIGqCKNQKX+H6x5G02+MZaY6/5AMF7EpwmilltNhxfASIUXQwt2PiyUS2Yhd2JtDrIzD0F3OBsLOAz2ZjQBHMARBUPJdNQaGIaT8JK3AhZhiWUwHPnAXiSMZao+ZMMXZQEF7QqAIrw0kWPZocBUtjH3niRFc7qgbJt8Mz6y25mngDfZExeJN0sYA+HcKow0T8sZqeU9gPjxZQ7pYPBv2EUQFBgPsuX+UCMxYIfO5F+TJNcPQgzgy2uCA2fny0Dwy7QyXV+5MD4OCHx0cHPX4v6grSQL7s9n476NFmDaCHSwoSY2KAJ53GpuLpO3gcDPLn/cHAGwz6w3f0cJrLH4OBPxgM5UMngB9bOng7OKSI7N8PeqdfT2vb5Na8dx0uc//76WAQvLv/Hf/1seGtBsGR8VPfJDq8IHthNG3W4Acytcl8hZ4n8flbwEt0uL9+rND6CfYe28S/3+733cc99/Gz8+irQ+QEDOagGfBkDAbT7f5u87+G73gcpUmhLQGoFmVXgLVXkaDYUr07OqBYrto7rZev4d0yLihqbMqF2HgB8jw5djFbAQIzo8bA9BhULtpKEv/YnBTlL4q4wOQggwTxYUQpHmBqzITAbFCv8Z+w+WfwjLrx98GgOXx+b42VCsEs0Ph9CW5v92j/EN2itbM9Q+i//AegzPXQh3b2MzmL2/enF1nMPxaFfvkls37m6rf/TvcCT/aYYTwASCxgv/Lv7nYarx4eZCdZTB6hJUjPNAaWs6rZENI8FqHRnU2Xxh+exQu0lhYRmUWduV0lk0LSHnWucM6ZeJgEKs1LOsOlqZaMJL+QyQCBeWyxJfeEMb6Sl9J8QDaF8TuDovO2Ebh+O2l10SYJadoF0FL7spqBt46NVzfDHzY1gwCNORjNwGYg7jEBJXeW59SkW4C5HDEltAJHpLYq38/jRVx02693dlQnYC3h//Y0H2GRm0h4pqmlIrBZdE4EnXgC12QofODCUkpkhS7PQwfVeg7Cu0Hxzu4NJ4JhtDjBMw0gPO12aG/mHIEVsaVkHBXXUZQook72dfhmaDrIGNklHgDQmtXVAiE4FDPAaF60RHjcCSf4Bw2FFkUP8igEAUGWbAC4ANVh37WiQC033GO9eeQnsXuVxlNxnoXjMeWWgkXMziMxTqe3IpdH4MkyI/A0yvjWXgRa76DUNnufSS5T3QMM/dvOztrO2OdgiH0qO7Z6ZNOjgtawGqlUhXI8OX0HVscCNbRb/EmcopVe4CFwud2hOA4d/oYgjrE9Bek4Tc84T+erorzlOQZmlVzmTBnZ8s8sYxvoFtDN+7/6ROr6jbdD5B/euy6wkHtgHkRqfc1C6v+T/cfAsfA870qmaRnv8xSRe8fVWKlPfFYZO1eOAiA3V1fBJkT0qGRDQvQrTiGu84vYqTsJW0Evp29cdZvEVGqlBP0ncYyJDi6jaEmbrYm2XdnzhjQNngOELAW6xxpOUFW5KSQVh+xyLLPLuOm67muS02UowXmWrpbeDmZlyuKlt+YQr6oWTMI8wsBwleYqqVcHr4hccAgk2rPm8zApSo/ys9lra9HDLiRrLaOCfNZRVn7O7bw66r+HNfaF+uXUuXLs7YWrKleXZ66yPTV7ebQq7fFqxQ1cnQxvRoD3q+T7fcfoHHXyNQvq+gi0RbAzItiw3TT+lCVoKT93SCjcUqxEyrUV/t1F4lUaovGs4NaQXKChGL5o/byzkc5pf4VD1Yp0KcKCGFKCzCodX8UpcCmdIgGk6Tkd9HGUFjqAZPU5XUqKxoynZI9XAjbJhMcfBAbnfYE5uX9/cLT38V4/fhJfxb74VXwRB+JInPm2yFyhcCxDq9cua8MOdWrrGbzqd7CUQsGSEf1R8iCBVCI2v5M4bCQMW/BF5krI1F8dEx0Nth6eTFwfkXXL+3EjiylVkby2Zl8ph9VKG86lnjaKp2VBb51Uh2GBqbaQLxYR6M9FBOQ/nP4BImuCiUfrzQPrhDyzZ69pz7JeaBbW7g3vkS9pXiiujwcnOxheGp5P4/xCeCffmq99dswPtvazQPyzefKtDVq85ViMZiayAW1OOw1o25J/Wjv2zsXiSv40VTumRkkwATmSDkXiTJC8Gc5gHCCh5RgymkyiF7ZmMsbAUsxid30RYdKiPI/HmAPNxB/JAuaUuJsmq3TWwcMuBhnJBDhqTCbqvHqLr9bjk1O2U60+GOAptequWaIl3BJj3NCM6iAqkY16TbZ5UapVMFirVa329LUxh0nJv7aOlCCgkmhePUPKa/0bCj2UmQYjLgAtjQAEjAs4fE5rzIlgsZzZEiUZaKEIFEd9baBiP0jBOMNGnXgz2JIblkkZ2VsXkZJtoivs8ySSTzfhAs+018o1g61JBggsrELCIppEI91adRRzPW20kUQsnkYcy8gltDaibYS+u39353kqLqD9OSYhRnrSEZK8KLry+VvzZUPKINY6kgdWoeO1bKissqr3iMxAZ4al6iNNEGz+jZad4O0ATx2h6Q0LlnRHp27Z/y77RX/7HbdsDcNFXCxtDtWPasuLzlNZmzXyJy6eU2PtslX5mmFmI9h5FGjh8BHjRV5n6PDF/SbuiA3hWTY0lrunz+7MmRZ2YGOEX/n4dalXG0I6Kkc6KSMBH7j7X2+85gQrNW85KDRhOo8SDHOPWSFfzqMiCpzs3ZZvYKpPZtrnLUvDWXMgkpwS9hE6Q4jQ19Zin75y99bA1ae1bMzDSqMG/r/OxKODAGRMZNcWqer7Bqj5/6hrZlM4x0bUWaGSd4e18nCCZ5CiqYpHonhuWjLXZ/Sy3mckyAL9ShwDTYgTUAQv0iRF79SkOQ7xNBSGUeXQPXYobYvfX4scWInAPCLnCTGYhKWllBIukLOHjo6Z9KbzVU6ThP8mI+wRYOLyNhlTFmHK8OikUySJENSjlrQsuv6kQfLx+GjUO9jodPiS3X/J0bJN1u/7U/zfoiDVZxC866P1dYhWIGX+MWb2s+PfKm4U22gP9Z8Nn9MMfzutliTI2/T59HD/YLS7ppvGLHCv7Q74CyYDfy3nsGlBxJjd0z6EB3oLjDmarNQZZ+3voJaq3h9jm7gHegqUl9qYTPBgkuP8IXsxYrGJaFJElqX8vhWrTvI9L8F6+R7EPZJySso2hhMgW4XPHSyyLf425DDcT6u8iFE+edRqumyQfCbtkku8s8FNIse2eUxmUoJVKI4IaxyQmCbN4cuq9IV7q6iRa6zQQx2zbnpT1BlQruHDt1OnQVWlvt1rlE+oAySCSAnk2kgc6FxYLSlX0tO6iHaE65rF0F1/qskaZVdVqTzjGOiOi2pC3VELk/6OoPG286yyuuXgd4ooJyilguTxkOFntORR6YyDlBtOdntn/6JN53jdLCPcO7Pf+DfvOPit9xz8tpxx1gbEQmYL3mcgVaNqgPcEvFNEhHvQK/fAshO+M7sRe8D78T5JkyZaseweWb1Il6RBRWjGRiM5gLy9j4nBcrsfjokAUGOwTe+BREDRcXQRzmf4K53RwENMhYHPWCaLyLqOi3xOcL4c759qKPD/cygPKsD9LM7y4j7HTGrT++IiztB2Pt0eDK6f34fz+T1lAyUTu/+cnfR7Xyu0D12KQIDvoKcP0jSPygS5He+w5Yc7uXwPFnVCQL0fBNSzAX3c79X1SLeZv+sDiGAwaHaQNQBI0NDOL+7Htz7739FNS97nv5Sg9tZC7f0oVCCvZ7uHnyskfRNr03ztL8R+dkHY6I0odGJ9LsFBctzbp8ABTDQ3lQYxPgnKyQNA6+JfcQJrHKtE1+Trnay0R3GeNsRFXOIantEQjOi/O/INHyE0cdmIhjU0sLbXAeuVgfU2A6tqGSzIjFCo8WyTdpqRIRnPYYK6Mvrcfbmzgz963bZxl3IYf0Q2G6rQKRlM4ROxNzmjlWMfJPDb8wgtl1TNCCg+NFtjrcFqNfRfhZwBGCzh99tDa1HtcyhOF3mlH++htLXlQNxhWtBU7tfW6d+YADl6Ba3d4ClOPDE0/N7B4Hm9VSnxwU2HAKoB8gDWjo+wuywRhDH64NR2C/IV3t5CAVjIwKyh/gxrng+ra0BURSnWCM5fPwjckbIftXB6T4TTq4NTGq3erZ2qkQ9QmoNdOWz0qr9DiwJ4zcla8FWbTtsOtuTNVRU4VneofGvo0Ahnt3F/NSSt37Nq4YTRc753i4zwDrROElobD/eGyZNRFmDtCeRTWBkeYFq334kOcmu+c4IBG/dUcDnAKPlEEOimicIJZxWHi2XpsnYy610Mpdq7tVX50NvuqBQgsj5av152stdk1uIjKA0KOy6vy69opdgddVT4eU/9lHG0J498r4lPr4nj3hyUXhu+zaHbWOPXmnn6tc/fh08L2naMKpY9xbakdDacEnkcv9V/JxQZ7fbKlXFLJ4mscwIe83kbpA4u/5Ujrv8qG6gLDcdi8itOKf/cFBtu4FKFX9dW4PMfMuT70TDxypEQTi7H8zXSZ9+lbUxdC2GjqzFzNazca3rKreuhpMWBcif1ZLZl/sO7k+1jkwuQ9BPpXFK1K32SxqHadDxrqvSXw/7uaIhR9fUfe6Nhw1kqCoq1LtRx7lH4AECarSbdU6GsNXKIubpHw0pjRV8aaLjKOXVOpQsl3JbtmBFhym+e/9f5g9jt3iEsHFEneDV7ED35oqde2FwKKuN5o+6dpyrBNKji/os21bBG6t4IgmYubpkzqOAC2YHWlq1KzNA9R462TBqn6sxir55oFpvRJJO3PmouZWCDnl4cg4rpRvPhxe04i6eclFAGShNG8v2BmCfdSl79wljuZOYxNPZdAxstoqRq/NrbPWENDe1Rg9XOzmSniX+mb7SFyn5J//5MDzN6mMmHmR18ijN+eAQ979rhpxiuyhqFDkilkt+AV2JJijnt3GHFh/s76NgDKZFoYmnL73dY9kED2uZnihx9yZGjZ8e/VYx5WIhwfPdT5Ru3GoPeNP6DbUa+hO1RcGqbMswPkvfVmoA0cjyf7lqN9oO0BnLx9nqTIldSdYJ3aDdUFsRODHr2+D4HzRrzx97juuHfdIxRa6A6Wzr8RTSfYuHrMKc6YcyPaD3hHyn+AXSJAZ2g7jTOUTu3YYTnWQRfYMdQWUwFNb0nZ99UmwL26oYOpT99emdpiYOB5/X/2/PRH3X3svHqNUzcgOduf/1cwIgpvISibu4BxieoJP+Cluv7HIljytu202fD7YaNUQbBrH6xSfY5R6lvlRbhuWxRjvRgox24PN6+WrXqRpnN0K7LOwIL4a7gSXnn6/pDZWpZ3yrNjazQ0c39t5zjVnvnwRmUg0kYsAN/Dvx3FLazd3i2iwcO7nf3MMwJ5ji5/4teY8rEpe/jNBcJCGlIlX5NZGrv6XdDfArkc5sPMHwSHhKidLEIfesGgT0oSF51X14UIGhQHfHPhviX8D7J11i/j8WG/PxBFdsFeeO9vE3g9Oz4ZIS72bk0VEUmWVcZmEeSw2VHCdHkZQa93S/q7Yfj3oHV4ZPewamuDqrcs/ef5ci/9NTfU/njVLV5ena4J0udfbSh7R7ty458KF+UcHTwTb46+PzxUPbmX/rV4e8HvYN9edsoi9NFFi8sLUOn8i3lE/EqCUVMaLplPy2FpFfD0XWmEa8m1YgJQ9dB6BWYtj1W3b1HHVeWWXkPnspsSHmmwsklX70rLRgYjI7nu4r0Mu++0mPOTZwOhbkLHVmHBd1wGGPrAdZ6nWuDDz2VDoaS6Ye5iGv76V/nFJln7oWjYa2SkkNDHQbXGXKwP05Smhvyu68SUBFQV0ZP/UoeiCxnoKGGxx2oMpSRHRzrX00DVjLTq7phBzQBadzXEAK6TzYRZj9tBgbdC2I8/1LgdHs4wlIsDIrxUIpOsKnFWg9T4YKegwZNB2ZjarEeoCfERg0Sl0ZSXMrbNY6lO0QcqRzSYqjrZS369qBCBMhcBzoS55u3N5Q6o45KSmnLudo6lKl6FvqUbp7ULyzghLMRdpEYst7ZhRWt/tL0KARuaWtly3cAZzcNPVjPQ2mlYQ+aTw17JJa4H9pDv6x2cpjszfoeWvNT3q56p1VtWzWzpYbLS/C8Ml3uqPYqnX9JozqovH/9g4OKk4TOlah4qtb6eCqvJoCcatP1V/gL05hox1rNuFU5tT1NNNbWJ8lX4O9DfU2MYl7N5+wFlEL2ehkEgIEEso1QqclSd/yNy/Io7tHU7n8XTr8aPhXuh41wdaAxhhc0tJ9YrZ7bSOlAxVLly7HOeBjx6a94nAO592CsRbYtfYylcilb+ejFkg69QH+qmHP5XUcueEiX8qwFgi3fBQav2jIwDO3NVbuqLe3CyPbefbr/JE9t+tpVxHZqBFbCjdptb5FFnHno3aX5ZfW1vYYOVHGYlC1pKMcSsHs69BcT0NXMFhJaG6PeDGtIbtUiCdp7Or+KRsxL8PD7PFzmMvvV2iCFkm3Y5MsqOxC8GyLXTQ89CU188P1SjjOEzubyX2gQzPUedZXbNtj1DuufBJs2hOKS1egXSpDAwycrghGlHzdh1tpbqzcTmGKWwdXNerkpTYZODMw37fBIRmjXyL+Hx5cNrdXANcvtJoUAFYhVK2rUWGmdzknGv1pSkKO7oK4h1kUqk8lG3wBNB6cYB+sMc2sMR8YuVzMn2v5GCR1qOi6lZyeViI0nnFLE3GfDeTBsQy59lz6hacUii3ky7NLlBqhmpVUXSp31kPCh/SZ/UEFr0bR7p7vTeTV9EGcn3bti2Xk5rZgP4YMeSid4yQZH7is9WsZD4++SI8fg9uAVe7kKinV/baEGLqHasdhFx8qqDZA0VL1byRCJFX1X5iUUwfw0ShSV1Z1Umu4ZWYPUVjTod2C2DOchBJWtdeqyOElst2I0yxb49fSWKdbXnMPgI+DU43mcX2CcYPVuBw5QnKH99Kr1DBTTeYFWLZP5GmjbwenZ7vvPh6cfD/ZHZx97tpXAzl6u3jkpzHXBNSnL1yczLycyV2/tbObuhYoyMbm2kdTkNS/lNLesE3uV+k52cw3USXHekGYEnS6GDh6o1CgquTcflS+l/aBE8YFLVzzViptCRZ5RtHPZ6cOQnI3LMo1sSoEljUJ2miurZk2YLFfQkdHmkVMDDRL7jgwchBS2+fJbF/U5/3NvBSx4YTJAf8Ekusk5YLS8DYzvQe0wQ1Kpsn8S71cxsFuUheZxAoxKY7CN1tsiVb4A5WTZNhGrtFnwQGwWXo9wELpwt5Qo0J3+hgkNRz4Q8w4pQ3AyozkfG6K0gTQZWkQZbF+8M0WVVQ5d3QPVnKFOvOnXJzB8oieUsVCez6nzgkrPjV+usl05EyXzvOkN61fPHCuuvJnwqqE5Z5ts2k93l/drZGcV780nuUtzR5nkYEr6w3qxGb3REnG9J97p6tcev3piB7FYbUxpmNzWaBpSbgfBlnYnWyOlEC/+IWOCq9VI06dvDVmxIUakma30/Tw16siwmh2QsmsZGZ1Guc1gfH+NZ1jtVHuL1KC7vLxV7a+GqN1ezL3JpK7hmrx8G5P+mQbY0VzpV8O9h6Tkt5V3nv9+0Dv8cHiwL94D+f98eHTA95/rG89VcetOZCA4i3CSpc0PLZI/MjwE5JW772Yxa4g3vgNNMRINidYFWNff//66/YaaNhD5SNso07V0dCvm/NrcsmgqqK//9vr1m9ZO6/XPf3vjY8QRSIk77YHtwn0vQelTfrpJGun6HklavjudktpUOsVBW0DKMkv2wprrVBRQ6whFQAnJRuzHLR3J8VmVqcTV4EeKralGgijhqS6XYQVrnEM6oCAjgR7P08klqQE7ZBXRupIrKtadRnHJ85RuS8dR6EsF5SgDGJSMYfKmMqOk7yrUJrKlfPmVjnfkKeDQMZamHlOTZSp2dzYdqEi6KJt+QyxzwBuKbETbkVTLlvhBUSAzICAuFZO7nMZqqvZqvxxAMjRH8iIamM29aJUk1DLmTPXhLcaCCgJYS2ByjuLLHyRDBt6T6M+Hr58/i5PDE0l7jr+c7PYO8bIclwDNnL1pSMZdZfM/U3v/2bATvJk9uEBMeIQF5s4a7iP196N5EXYqR1bVWqB6uQYY0KGNfd0uNcZ9tTYYrTKRIfrl1xygyzWKjW8VvWHCxa/LC3ASZU0+/oWe5DCLc9smUHeZlcTocUNckDG/NCAMt2IUsN+sV8UxwWmrDZr4hxYsZf/ZjGYdo3aa/xB3F/aL+7ImfnZCVc5Ong11cX64Fx/42wf7Gz1Y6jnSa3P02ugwLHXiVbvRIs1ulRlMHeRk7CEmbp15wwMuGvHlBVxI9NdzjDXhTbQu6GfrGMaAmSpR2aLTbYpFBmI/tY7goXs1i1DBcOaJDkxbQUHiNire0oi19KKb4ZuhV/IAJjCn+bwEa5pKXsaZNGT+yMCa1OqNVe7I9CamACRKLwpSrNgj9GO13ursFHcb8HhA5belrkAvLpB7Xoe5yMMrvrYYUV5HLplOuTFXP2+OuXIS9zlbUEzmUZisltIGkVNvdeCVnkaUYc1loDJZ0ZqjhDKr52Crwob5xjutbVpFa5XZTYWlgrupyLm+obKuxGbFu1RrkIz+P8nK6CZkPIuyBR7yjBKVxy1CTV2Eq2mMqXLCggLfELs5P0mT1H7NJ5FF4xXLML4jvH32oEdcyhiMYnlQZQyYKe0a1zKsU/6dyL9z+TdUuQ2s1AbaZqEOw4ACm0nI+Qp6FmP+j6nJrHDTzMMMOjm5sCo7uRjc5AulzAxOlqlSUioFzTqzWU0wNY0mUai/WqfVCIAORRlhBrmRyVzhBL9n4TRe5d03O0b1P5emYvscRJPLVbIhRNvyg3vKBq9i8yQy6twDBNikKTSbaJSvshlGqlTjZSQ8djV1olJkxSADutYI3nY8vz+8e2hybk+mJtZ9aCOiCSPGLd2Y/NugME3VXqKukiZxmUtYWThcF/N1mk1l5jl0JOK1lLXu3U2xmOzplS1pZxixMN0T2AIW4levQ6GLIEz2rR7zolkWkpMzNzsMwyZVWoQmkAlOeRroNsueaqQW9/v3e/ef/XffYI+TW9V0LMgiwLhJpO7aQb9k3SWqTv8w5QRMuLSLJHhKnNaFAdAJU0p9XH7mw+SSFLgxNThkMvRh5i1gFDnmFYSH+fICOEART2ilRIoDDKEpuiCSr4QLKvdg8KJSyqobUD6WiO0NfuujCvKy/sLaTWP8sTHtCrRhzmViWuSFYhxhokjYR0W8QK4oPWGpmLI8IqWQuLDhLEDoAbUYxcqLkLIp4uTggIJKXJDCeInmON5XTxivfIcHr/WewwP/KJex7cWzN5n0a93ZCc/+5+r1d6rY1jWirMTp9Ec8i/YbSwm3gx5+QAuv08TJlRdRpJmWGfp2WjmcrzVK+ppDKJxnolZZdxX2aL5BZV8f6fCI6s4pMwiV6AjhJkJvj0El8JnDRGQxnoSWydH0Tbg5CgZvdpqTixAvDI4yKyMTGQ+DWhvo97FAF90lMlQtB+uyqdoUgObWtDtCWbR2nlVHn86qvrOnT+ktOfTXkqeadi3Km5hUj9aS0P0GlFyL0qnZ6l5QB+7sIs4xJQFGwo/p3lU622L0gmD91D1hjV/u/O/MXr2NfY3pqt6q7tiycMc7JGBd6Nd6u9amHkvwj9i2qvYtm3g/at0iquXejv5Eu5axaC3JSeVeoeOwHC4gK0rzTEMvpra+kG5kuNGILQ3q/h1K0WMYtlIqyrtWC+WO0KJf2ly++rK0lxrGyzniMYz0IEZyFCMzjJHhQGXGSh2Xk1IzxD59R2eLV2pIU94RpimxLPiECI7laKZthncE7kGbCO37sb9UPRMbPRJ2XTLuiau8qtR3SqnqNgGuM/Q90rJt1kKTw0n3bnTdf6YjR56Zs234nsNNntWfb/vQ4kLGcnYmwbFx7IN8YtvYhyP5dOQax+xp2dWGR4UU6MD5Q4Z4bDk4Mth6z0jCH9bYHaum3B/0Ix2l8szadWQfWqsctHv5SP6pD2QJA94BC4/qCjzgOiAvd68keVFggkYTB8H3kexruxheQnJr3PbUOTL7fDg82v1MyujBP/c+fz09PD5yDqWEeZ5OYsqOmqUzUI1SO/QBbQqCzAnqzQI1/ngeW6/S8Tz+98re+flqzLdPFw7pQA5IBxkpkCieG01d0YYRh8di+o667byBpgzd+9Qo/IZA4FzS4tpm07VXFlqCKuWT67A3v1bUlgnnHHnbbqRW3M6iRXoVTWXMXDV9nbmMjfpUI0FLaVlKynzPXNVhPqyIx9JVXncvmC068411/c7L4TqRtmyP8GtkAq/2uD7ne9N32tTFi1fNDhRcXYPFj8sHcq51OKqeYxquMjuUmH7NNbVqXhUgJ3S39tI/jkeoceQ3VKfwK4a+KDQt2fUbwsJgjTK1uC35mLuDjP2Ve+o29IMutg29rfO4aYo6a9X5bBz6qnpIRWs7Wy5/vaGgYuKWA+33ir3c8sNZPa3xvTGRBphNOmHtRA7cqW4/1Wcn/9P1bK4NULarYHpy+S2F1oBBu0UFU3zHn5e4DH4Nf1/L3iVrr3B2h7E7fN1m68QQh/8HGbjvfQ=="
_setup_cells = json.loads(zlib.decompress(base64.b64decode(_SETUP_PAYLOAD)).decode("utf-8"))

print("=== Restoring notebook setup and exact rule definitions ===")
for _entry in _setup_cells:
    print(f"\n--- Restoring source cell {_entry['cell']} ---")
    exec(compile(_entry["source"], f"<original_notebook_cell_{_entry['cell']}>", "exec"), globals())

# Correct the GPT-2 special-token config warnings. This changes no weights/vocab.
try:
    champion_model.encoder.config.bos_token_id = None
    champion_model.encoder.config.eos_token_id = None
    champion_model.encoder.config.pad_token_id = None
except Exception:
    pass

# One verified threshold dictionary, independent of any later exploratory champ_thr.
EXP15_THRESHOLDS = {
    "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
    "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
    "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35,
}
EXP15_REFERENCE = 0.333307
EXP15_EXCLUSIONS = {
    "associate professor", "both pw", "manchili",
    "oblique", "subsequently", "the police patil",
}
EXP15_TRIM_CHARS = " \t\r\n,.;:()[]{}"

# Verify training/validation isolation.
assert set(train_texts).isdisjoint(val_texts), "Train/validation document IDs overlap."
assert set(train_texts) == set(train_gold)
assert set(val_texts) == set(val_gold)
assert len(train_texts) == 35 and len(val_texts) == 35
assert len(BIO_LABELS) == 25

def exp15_full_pipeline(model, texts, use_gold_metrics=True):
    # Rebuild all model-dependent predictions, then apply the original fold-1 rules.
    raw = predict_texts(model, texts)
    filtered = apply_thr(raw, EXP15_THRESHOLDS)
    base = merge_all_original(texts, filtered, keep)

    result = {
        str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
        for d, spans in base.items()
    }

    # Apply the validated name_shape_plus_context WIT candidate procedure.
    for did, candidates in val_rule_candidates.items():
        did = str(did)
        if did not in texts:
            continue
        text = texts[did]
        pred = result.setdefault(did, set())

        for s, e, label in candidates:
            if str(label) != "WIT":
                continue
            s, e = int(s), int(e)
            surface = _candidate_surface(text, s, e)

            if not _near_identifier(text, s, e, radius=60):
                continue
            if not _valid_person_surface(surface, "name_shape_plus_context"):
                continue
            if not _near_identifier(text, s, e, radius=30):
                continue
            if surface.casefold() in EXP15_EXCLUSIONS:
                continue
            if any(max(s, ps) < min(e, pe) for ps, pe, _ in pred):
                continue

            pred.add((s, e, "WIT"))

    # Apply the selected CASENO + STAT boundary correction.
    final = {}
    for did, spans in result.items():
        text = texts[did]
        cleaned = set()
        for s, e, lab in spans:
            if lab in ("CASENO", "STAT"):
                while s < e and text[s] in EXP15_TRIM_CHARS:
                    s += 1
                while e > s and text[e-1] in EXP15_TRIM_CHARS:
                    e -= 1
            if s < e:
                cleaned.add((int(s), int(e), str(lab)))
        final[did] = sorted(cleaned)

    if use_gold_metrics:
        metrics = calculate_exact_span_metrics(val_gold, final, ENTITY_LABELS)
        return final, metrics
    return final

# Reproduce the known validation baseline before training.
champion_model.eval()
exp15_baseline_predictions, exp15_baseline_metrics = exp15_full_pipeline(
    champion_model, val_texts, use_gold_metrics=True
)
exp15_baseline_score = float(exp15_baseline_metrics["MACRO_F1"])

print("\n=== PRE-TRAIN VALIDATION CHECK ===")
print(f"Rebuilt WIT + CASENO/STAT hybrid: {exp15_baseline_score:.6f}")
print(f"Expected reference:              {EXP15_REFERENCE:.6f}")

if abs(exp15_baseline_score - EXP15_REFERENCE) > 0.002:
    raise RuntimeError(
        "The rebuilt validation pipeline does not match 0.333307 closely enough. "
        "Stopping before training so the comparison cannot be misleading."
    )

# Build BIO training chunks from character-offset gold spans.
MAX_LEN = 512
STRIDE = 448
BATCH_SIZE = 1
EPOCHS = 1
ENCODER_LR = 5e-6
CLASSIFIER_LR = 2e-5
WEIGHT_DECAY = 0.01

def exp15_encode_doc(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)
    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            continue
        token_ix = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not token_ix:
            continue
        for j, i in enumerate(token_ix):
            tag = ("B-" if j == 0 else "I-") + lab
            tid = label2id[tag]
            if tags[i] != label2id["O"] and tags[i] != tid:
                conflicts[i] = True
            else:
                tags[i] = tid

    for i, bad in enumerate(conflicts):
        if bad:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break
        chunk_labels = []
        for global_i in range(start, end):
            if global_i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[global_i])
                seen.add(global_i)
        chunks.append({"input_ids": ids[start:end], "labels": chunk_labels})
        if end == len(ids):
            break
    return chunks, tags

exp15_chunks = []
exp15_counts = torch.zeros(len(BIO_LABELS), dtype=torch.long)
for did, text in train_texts.items():
    chunks, full_tags = exp15_encode_doc(text, train_gold[did])
    exp15_chunks.extend(chunks)
    for tag in full_tags:
        if tag >= 0:
            exp15_counts[tag] += 1

assert exp15_chunks, "No training chunks were produced."
print("\nTraining documents:", len(train_texts))
print("Training chunks:", len(exp15_chunks))

# Conservative inverse-frequency weighting; bounded to avoid huge rare-tag weights.
exp15_weights = exp15_counts.float().clamp_min(1).pow(-0.25)
exp15_weights = (exp15_weights / exp15_weights.mean()).clamp(0.5, 3.0).to(device)

def exp15_collate(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    ids = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, item in enumerate(batch):
        n = len(item["input_ids"])
        ids[i, :n] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"], dtype=torch.long)
    return ids, mask, labels

from torch.utils.data import DataLoader
exp15_loader = DataLoader(
    exp15_chunks, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=exp15_collate
)

# Fine-tune an independent copy; champion_model itself is never optimized.
exp15_model = copy.deepcopy(champion_model).to(device)
exp15_model.train()
exp15_optimizer = torch.optim.AdamW(
    [
        {"params": exp15_model.encoder.parameters(), "lr": ENCODER_LR},
        {"params": exp15_model.classifier.parameters(), "lr": CLASSIFIER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
exp15_loss_fn = nn.CrossEntropyLoss(
    weight=exp15_weights, ignore_index=-100
)

for epoch in range(EPOCHS):
    exp15_model.train()
    losses = []
    for ids, mask, labels in tqdm(exp15_loader, desc=f"Weighted fine-tune {epoch+1}/{EPOCHS}"):
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        exp15_optimizer.zero_grad(set_to_none=True)
        logits = exp15_model(ids, mask)
        loss = exp15_loss_fn(
            logits.reshape(-1, len(BIO_LABELS)), labels.reshape(-1)
        )
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss; stopping this experiment.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(exp15_model.parameters(), 1.0)
        exp15_optimizer.step()
        losses.append(float(loss.item()))
    print(f"Epoch {epoch+1} mean weighted loss: {np.mean(losses):.5f}")

# Fair comparison with exactly the same full validation pipeline.
exp15_model.eval()
exp15_ft_predictions, exp15_ft_metrics = exp15_full_pipeline(
    exp15_model, val_texts, use_gold_metrics=True
)
exp15_ft_score = float(exp15_ft_metrics["MACRO_F1"])

print("\n=== EXPERIMENT 15 RESULTS ===")
print(f"Champion hybrid:   {exp15_baseline_score:.6f}")
print(f"Fine-tuned hybrid: {exp15_ft_score:.6f}")
print(f"Delta:             {exp15_ft_score-exp15_baseline_score:+.6f}")
print("\nPer-label F1:")
for lab in ENTITY_LABELS:
    old = exp15_baseline_metrics[lab]["f1"]
    new = exp15_ft_metrics[lab]["f1"]
    print(f"{lab:12s} {old:.4f} -> {new:.4f} ({new-old:+.4f})")

# Save a separate candidate checkpoint only if it beats the reference.
if exp15_ft_score > EXP15_REFERENCE:
    exp15_out = Path("/kaggle/working/experiment15_weighted_finetune.pt")
    torch.save({
        "model_state_dict": exp15_model.state_dict(),
        "bio_labels": BIO_LABELS,
        "experiment": "class_weighted_finetune_fold1",
        "epochs": EPOCHS,
        "encoder_lr": ENCODER_LR,
        "classifier_lr": CLASSIFIER_LR,
        "validation_macro_f1": exp15_ft_score,
        "reference_hybrid_macro_f1": EXP15_REFERENCE,
    }, exp15_out)
    print("\nSaved separate candidate checkpoint:", exp15_out)
else:
    print("\nRejected: did not beat 0.333307; no checkpoint saved.")

print("Champion model weights and existing prediction files remain untouched.")
print("No test gold labels were used.")
